# 933점 이후 — 하이퍼파라미터 튜닝 + fold 10 + seed 앙상블

## 현재까지 경과

| 버전 | 리더보드 | 핵심 |
|---|---|---|
| 818점 | 818.84 | `TimeSeriesSplit` fold 5 단독 |
| 897점 | 897.07 | `StratifiedKFold` 5-fold, 트랙맨 `exact`(test에서 0) |
| **933점** | **932.96** | 트랙맨 `asof` 모드 (2025가 2024 값을 받음) — **현재 기준선** |
| (실패) | 930.00 | 타자 측 트랙맨 피처 추가 — 롤백됨 |

933점 구성(트랙맨 `asof`, CatBoost 단독, 5-fold)은 그대로 두고, 이번엔
**모델 자체를 더 잘 뽑아내는 방향**으로 개선한다.

## 이 노트북에서 하는 것

1. **Optuna 하이퍼파라미터 탐색** — `depth=6, lr=0.05`는 한 번도 튜닝 안 한 기본값이었다.
   빠른 3-fold 서브샘플 채점으로 40회 탐색한다.
2. **fold 5 → 10** — 각 fold가 90%를 학습하게 되어, 개별 모델 품질이 오르고
   평균 대상도 늘어 분산이 더 줄어든다.
3. **seed 앙상블 (3개)** — 같은 설정을 다른 무작위 분할로 3번 반복해서 평균낸다.
   총 10 × 3 = **30개 모델**을 학습해서 평균한다.

셋 다 **933점 대비 구조를 안 바꾸고 안전하게 쌓는 개선**이다 (새로운 피처를
추가하는 게 아니라, 같은 정보를 더 안정적으로 뽑아내는 것).

## ⚠️ 검증에 대한 경고 (이전과 동일)

`StratifiedKFold`를 쓰므로 로컬 홀드아웃 검증은 성능 지표로 쓸 수 없다.
다만 **Optuna 튜닝(Cell 6a)만은 예외**다 — "이 하이퍼파라미터가 주어진 데이터를
얼마나 잘 맞히는가"는 `StratifiedKFold`로도 정직하게 잴 수 있으므로, 이 부분은
로컬 판단이 유효하다. Cell 6b 이후(fold 10 × seed 3 최종 학습)의 OOF와
933점 대비 실제 개선 여부는 **리더보드로만** 판단한다.


## [Cell 0] 라이브러리 및 설정

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ---------------- 설정 ----------------
# 'asof'  : merge_asof backward. 2025 test 행이 가장 최근(2024) 트랙맨 값을 받는다.
#           학습/추론이 동일한 규칙을 쓰므로 원칙적으로 더 타당하다.
# 'exact' : (season, month) 정확 일치 + fillna(0). 900점 버전과 완전히 동일한 동작
#           (트랙맨에 2025가 없어 test에서는 전부 0이 된다).
TRACKMAN_MODE = 'asof'

N_SPLITS = 10                 # 5 -> 10 (각 fold가 90%를 학습, 평균 대상도 늘어 분산 감소)
SEEDS = [42, 202, 2024]       # seed 앙상블 (3개) -> 총 N_SPLITS * len(SEEDS) = 30개 모델
N_OPTUNA_TRIALS = 40          # 하이퍼파라미터 탐색 횟수

# --- 2026-08-18 추가 (2024 시즌 홀드아웃 3-seed 짝지어 검증 결과 반영) ---
# 조건부 투수통계: 기준선 대비 +20~27점 (3 seed 전부 우세, 분산도 ±18->±6로 감소)
USE_COND_STATS = True
# 재중심화: 11개 설정 전부에서 +11~22점 (평균 +18)
RECENTER = True
HOLDOUT_SEASON = 2024         # 오프셋 측정용 홀드아웃 시즌 (이 시즌은 학습에서 빼고 1회 측정)
N_HOLDOUT_FOLDS = 3
# 죽은 피처: asof_pitcher_n이 '경기내'가 아니라 '커리어 누적'이라 의도대로 동작하지 않음
#   is_long_relief 는 전체의 86%(이닝>1 중 97%)로 사실상 inning>1 과 동일,
#   is_strict_inherited_runner 는 0.05%로 상수, pitches_per_inning 은 커리어투구수/이닝.
#   (효과는 +4점 수준으로 미미하나 코드 정합성 차원에서 제거)
DEAD_FEATURES = ['is_long_relief', 'is_short_relief',
                 'is_strict_inherited_runner', 'pitches_per_inning']
print(f"TRACKMAN_MODE = {TRACKMAN_MODE} | N_SPLITS = {N_SPLITS} | SEEDS = {SEEDS}")

# v5: Optuna 재탐색을 끈다. 다시 탐색하면 파라미터가 바뀌어 리더보드 차이가
# '트랙맨 v2 효과'인지 '파라미터 변화'인지 구분되지 않는다 (v4 때 실제로 겪음).
# 아래는 v4(987.3936) 실행에서 나온 값 그대로.
RUN_OPTUNA = False
V4_BEST_PARAMS = {
    "learning_rate": 0.022831883708228414,
    "depth": 8,
    "l2_leaf_reg": 8.552069332567962,
    "bagging_temperature": 0.05636104060100738,
    "random_strength": 0.7731135614050382
}

# v6(릴리스 동역학 12개)는 리더보드 988.4720 으로 v5(990.9528) 대비 -2.48 -> 기각.
# 코드는 보존하되 기본 False. 스크리닝 +20 / 누수없는 홀드아웃 +5 / 실측 -2.48 이었다.
USE_RELEASE_DYNAMICS = False

# --- v8 절개 실험 (m): 네 항목 중 하나만 켠다. 나머지 셋은 v5 와 동일해진다. ---
DROP_CAL = ['game_month', 'game_dayofweek']
COND_DECAY = 1.0          # 1.0 이면 감쇠 없음 = v5 와 수식적으로 동일
USE_REST_FOUL = False
USE_COND_PB = False


## [Cell 1] Step 1~14 — 학습/추론 공용 소스

`STEPS_SRC` 문자열 하나를 노트북과 `script.py`가 공유한다.
전처리가 두 갈래로 갈라지는 것 자체를 불가능하게 만드는 장치
(이전에 이 불일치로 0점이 난 적이 있다).


In [ ]:
STEPS_SRC = r"""
def step1_basic_features(df):
    df_proc = df.copy()
    df_proc['is_weekend_day_game'] = np.where(
        (df_proc['game_month'].isin([4, 5, 9, 10])) & (df_proc['game_dayofweek'].isin([5, 6])), 1.0, 0.0)
    df_proc['is_heat_wave_game'] = np.where(df_proc['game_month'].isin([7, 8]), 1.0, 0.0)
    return df_proc


def step2_pitcher_role_features(df):
    df_proc = df.copy()
    df_proc['is_pure_starter'] = np.where(df_proc['inning'] == 1, 1.0, 0.0)
    df_proc['is_long_relief'] = np.where(
        (df_proc['inning'] > 1) & (df_proc['asof_pitcher_n'] >= (df_proc['inning'] - 1) * 12), 1.0, 0.0)
    df_proc['is_short_relief'] = np.where(
        (df_proc['inning'] > 1) & (df_proc['asof_pitcher_n'] < (df_proc['inning'] - 1) * 12), 1.0, 0.0)
    return df_proc


def step3_matchup_features(df):
    df_proc = df.copy()
    if 'pitcher_hand' in df_proc.columns and 'batter_hand' in df_proc.columns:
        df_proc['is_same_hand'] = np.where(df_proc['pitcher_hand'] == df_proc['batter_hand'], 1.0, 0.0)
    return df_proc


def step4_refined_count_features(df):
    df_proc = df.copy()
    b, s = df_proc['balls_before'], df_proc['strikes_before']
    df_proc['is_first_pitch'] = np.where((b == 0) & (s == 0), 1.0, 0.0)
    df_proc['is_full_count'] = np.where((b == 3) & (s == 2), 1.0, 0.0)
    pitcher_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
    batter_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
    neutral = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
    df_proc['count_advantage'] = np.select(
        [pitcher_ahead, batter_ahead, neutral], ['Pitcher', 'Batter', 'Neutral'], default='None')
    df_proc['is_waste_pitch_sit'] = np.where(((b == 0) & (s == 2)) | ((b == 1) & (s == 2)), 1.0, 0.0)
    df_proc['is_must_strike_sit'] = np.where(((b == 3) & (s == 0)) | ((b == 3) & (s == 1)), 1.0, 0.0)
    return df_proc


def step5_pitches_per_inning(df):
    df_proc = df.copy()
    df_proc['pitches_per_inning'] = df_proc['asof_pitcher_n'] / df_proc['inning'].clip(lower=1)
    return df_proc


def step6_combined_runner_features(df):
    df_proc = df.copy()
    df_proc['is_risp'] = df_proc['base_state'].astype(str).apply(
        lambda x: 1.0 if ('2' in x) or ('3' in x) else 0.0)
    df_proc['is_strict_inherited_runner'] = np.where(
        (df_proc['inning'] > 1) & (df_proc['asof_pitcher_n'] < 5) & (df_proc['num_runners_on'] > 0), 1.0, 0.0)
    df_proc['is_self_risp'] = np.where(
        (df_proc['asof_pitcher_n'] >= 15) & (df_proc['is_risp'] == 1.0), 1.0, 0.0)
    li_filled = df_proc['li'].fillna(0)
    df_proc['risp_pressure_index'] = df_proc['is_risp'] * li_filled
    df_proc['is_steal_threat_sit'] = np.where(
        (df_proc['runner_on_1b'] == 1) & (df_proc['runner_on_2b'] == 0)
        & (df_proc['score_diff_pitcher_team'].abs() <= 3), 1.0, 0.0)
    return df_proc


def step7_bayesian_smoothing(df, prior_mean=0.64):
    df_proc = df.copy()
    C = 50
    if 'asof_pitcher_success_rate' in df_proc.columns and 'asof_pitcher_n' in df_proc.columns:
        n = df_proc['asof_pitcher_n']
        curr = df_proc['asof_pitcher_success_rate']
        df_proc['smoothed_pitcher_success_rate'] = (n * curr + C * prior_mean) / (n + C)
    return df_proc


def step8_batter_toughness_features(df):
    df_proc = df.copy()
    if 'asof_batter_success_rate' in df_proc.columns and 'asof_batter_middle_rate' in df_proc.columns:
        df_proc['tough_batter_index'] = (1.0 - df_proc['asof_batter_success_rate']) * (1.0 - df_proc['asof_batter_middle_rate'])
    return df_proc


def step9_garbage_time_features(df):
    df_proc = df.copy()
    df_proc['is_garbage_time'] = np.where(df_proc['score_diff_pitcher_team'].abs() >= 7, 1.0, 0.0)
    df_proc['garbage_time_index'] = df_proc['score_diff_pitcher_team'].abs() / (10 - df_proc['inning']).clip(lower=1)
    return df_proc


def step10_recent_form_momentum(df):
    df_proc = df.copy()
    tc = ['asof_pitcher_prev1_game_success_rate',
          'asof_pitcher_prev3_game_success_rate',
          'asof_pitcher_prev5_game_success_rate']
    if all(c in df_proc.columns for c in tc):
        p1, p3, p5 = df_proc[tc[0]], df_proc[tc[1]], df_proc[tc[2]]
        df_proc['momentum_short'] = p1 - p3
        df_proc['momentum_mid'] = p1 - p5
        df_proc['is_heating_up'] = np.where((p1 > p3) & (p3 > p5), 1.0, 0.0)
        df_proc['is_cooling_down'] = np.where((p1 < p3) & (p3 < p5), 1.0, 0.0)
    return df_proc


def step11_veteran_and_pressure_features(df):
    df_proc = df.copy()
    df_proc['is_rookie'] = np.where(df_proc['asof_pitcher_n'] < 684, 1.0, 0.0)
    df_proc['is_veteran'] = np.where(df_proc['asof_pitcher_n'] > 3725, 1.0, 0.0)
    li_filled = df_proc['li'].fillna(0)
    df_proc['rookie_crisis_risk'] = df_proc['is_rookie'] * li_filled
    df_proc['veteran_clutch_ability'] = df_proc['is_veteran'] * li_filled
    return df_proc


def step12_first_pitch_tendency(df):
    df_proc = df.copy()
    if 'asof_pitcher_fastball_rate' in df_proc.columns and 'asof_pitcher_strike_rate' in df_proc.columns:
        if 'is_first_pitch' in df_proc.columns:
            df_proc['first_pitch_fastball_strike_idx'] = (
                df_proc['is_first_pitch'] * df_proc['asof_pitcher_fastball_rate'] * df_proc['asof_pitcher_strike_rate'])
    return df_proc


def step13_sac_fly_threat(df):
    df_proc = df.copy()
    is_3b = df_proc['base_state'].astype(str).apply(lambda x: 1.0 if '3' in x else 0.0)
    df_proc['is_sac_fly_threat'] = np.where(
        (is_3b == 1.0) & (df_proc['outs_before'] < 2)
        & (df_proc['score_diff_pitcher_team'].abs() <= 3), 1.0, 0.0)
    return df_proc


def step14_convert_to_category(df):
    df_proc = df.copy()
    original_cat_cols = ['pitcher_id', 'batter_id', 'pitcher_team_id', 'batter_team_id',
                         'pitcher_hand', 'batter_hand', 'base_state', 'stadium',
                         'pitch_name', 'top_bottom', 'game_type']
    created_cat_cols = ['is_weekend_day_game', 'is_heat_wave_game', 'is_pure_starter',
                        'is_long_relief', 'is_short_relief', 'is_same_hand', 'is_first_pitch',
                        'is_full_count', 'count_advantage', 'is_waste_pitch_sit',
                        'is_must_strike_sit', 'is_risp', 'is_strict_inherited_runner',
                        'is_self_risp', 'is_steal_threat_sit', 'is_sac_fly_threat',
                        'is_garbage_time', 'is_rookie', 'is_veteran',
                        'is_heating_up', 'is_cooling_down']
    all_cat_cols = [c for c in original_cat_cols + created_cat_cols if c in df_proc.columns]
    for c in all_cat_cols:
        df_proc[c] = df_proc[c].astype('category')
    return df_proc
"""

exec(STEPS_SRC)
print("step1~14 정의 완료")


## [Cell 2] Step 15~18 — Trackman 요약 (학습 시 1회만 실행)

In [ ]:
MAPPING_SRC = r"""
# pitcher_id <-> pitcher_trackman_id 매핑 재구축.
# 주최측이 준 pitcher_id_mapping.csv 는 구종비율 하나로만 매칭돼 약 91%가 틀렸다
# (시즌간 일관성 1.9%, 2024 커버리지 28%). 여기서 다시 만든다.
#   1단계 팀   : (월 x 요일 x 공수) 63차원 투구량 프로파일 -> 헝가리안.
#                검증 = 10개 팀이 6시즌 내내 같은 프랜차이즈로 대응되는가 (10/10).
#                ※ 월 단위 9차원으로는 실패한다 - 팀별 월간 분포가 거의 같아 비용이 평평해진다.
#   2단계 투수 : 팀-시즌 안에서 등판 프로파일 + 이닝 분포 + 구종배합 + 총투구량. 손은 하드제약.
#                검증 = 교정 전 시즌간 일관성 90.9% (매칭에 시즌간 정보를 안 쓰므로 순환 아님).
# 이 문자열이 단일 소스다. tools/rebuild_pitcher_mapping.py 가 노트북에서 이걸 읽어 쓴다.
from scipy.optimize import linear_sum_assignment

_MINOR_PREFIX = ('MIN_', 'KBO_', 'ACE_')   # 2군 / 올스타 / 기타


def _mp_prep(train_df, trackman_df):
    tr = train_df[['season', 'game_month', 'game_dayofweek', 'inning', 'top_bottom',
                   'pitcher_id', 'pitcher_hand', 'pitcher_team_id', 'asof_pitcher_pitchmix_n',
                   'asof_pitcher_fastball_rate', 'asof_pitcher_breaking_rate',
                   'asof_pitcher_offspeed_rate']].copy()
    tm = trackman_df[['season', 'game_month', 'game_dayofweek', 'inning', 'top_bottom',
                      'pitcher_trackman_id', 'pitcher_hand', 'pitcher_team',
                      'pitch_type_group']].copy()
    # 손 코딩이 다르다: train 은 1=Left/2=Right 정수, trackman 은 'Left'/'Right' 문자열
    tr['pitcher_hand'] = tr['pitcher_hand'].map({1: 'L', 2: 'R'})
    tm['pitcher_hand'] = tm['pitcher_hand'].map({'Left': 'L', 'Right': 'R'})
    tr['tb'] = tr['top_bottom']
    tm['tb'] = tm['top_bottom'].map({'Top': 'T', 'Bottom': 'B'})
    tm['grp'] = tm['pitch_type_group'].astype(str).str.lower()
    tm['team'] = tm['pitcher_team'].replace({'SK_WYV': 'SSG_LAN'})   # 2021 개명, 같은 프랜차이즈
    tm['is_major'] = ~tm['pitcher_team'].str.startswith(_MINOR_PREFIX, na=False)
    return tr, tm


def _mp_cells(df, key):
    d = df.assign(c=df['game_month'].astype(str) + '_' +
                    df['game_dayofweek'].astype(str) + '_' + df['tb'])
    return d.pivot_table(index=key, columns='c', aggfunc='size', fill_value=0).astype(float)


def _mp_unit(X):
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-9)


def _mp_match_teams(tr, tm, seasons):
    major = tm[tm['is_major']]
    rows = []
    for s in seasons:
        pa = _mp_cells(tr[tr['season'] == s], 'pitcher_team_id')
        pb = _mp_cells(major[major['season'] == s], 'team')
        pa, pb = pa.div(pa.sum(1), axis=0), pb.div(pb.sum(1), axis=0)
        cols = sorted(set(pa.columns) & set(pb.columns))
        A, B = pa[cols].values, pb[cols].values
        C = ((A[:, None, :] - B[None, :, :]) ** 2).sum(-1)
        r, c = linear_sum_assignment(C)
        rows += [dict(season=s, tid=pa.index[i], code=pb.index[j]) for i, j in zip(r, c)]
    piv = pd.DataFrame(rows).pivot(index='tid', columns='season', values='code')
    stable = int((piv.nunique(axis=1) == 1).sum())
    print(f"  [팀] 6시즌 내내 동일 프랜차이즈: {stable}/{len(piv)}")
    if stable != len(piv):
        raise RuntimeError("팀 매칭이 시즌 간 불일치.\n" + piv.to_string())
    return piv.iloc[:, 0].to_dict()


def _mp_train_mix(sub):
    '''train 의 누적 asof 비율에서 그 시즌만의 구종배합을 복원'''
    g = sub.sort_values('asof_pitcher_pitchmix_n').groupby('pitcher_id')
    n0 = g['asof_pitcher_pitchmix_n'].first()
    n1 = g['asof_pitcher_pitchmix_n'].last()
    out = {c: g[col].last() * n1 - g[col].first() * n0 for c, col in
           [('fastball', 'asof_pitcher_fastball_rate'),
            ('breaking', 'asof_pitcher_breaking_rate'),
            ('offspeed', 'asof_pitcher_offspeed_rate')]}
    M = pd.DataFrame(out)
    return M.div(M.sum(1).replace(0, np.nan), axis=0)


def build_pitcher_map(train_df, trackman_df):
    tr, tm = _mp_prep(train_df, trackman_df)
    seasons = sorted(tr['season'].unique())
    team_of = _mp_match_teams(tr, tm, seasons)
    tr = tr.assign(team=tr['pitcher_team_id'].map(team_of))
    major = tm[tm['is_major']]
    mixsrc = tm[tm['grp'].isin(['fastball', 'breaking', 'offspeed'])]  # 배합은 2군 포함
    MIX = ['fastball', 'breaking', 'offspeed']
    rows = []
    for s in seasons:
        a_all, b_all = tr[tr['season'] == s], major[major['season'] == s]
        mix_a = _mp_train_mix(a_all)
        ms = mixsrc[mixsrc['season'] == s]
        mix_b = pd.crosstab(ms['pitcher_trackman_id'], ms['grp'], normalize='index')
        for team in sorted(set(team_of.values())):
            a, b = a_all[a_all['team'] == team], b_all[b_all['team'] == team]
            if a.empty or b.empty:
                continue
            Pa, Pb = _mp_cells(a, 'pitcher_id'), _mp_cells(b, 'pitcher_trackman_id')
            Ia = a.assign(i=a['inning'].clip(1, 10)).pivot_table(
                index='pitcher_id', columns='i', aggfunc='size', fill_value=0
                ).reindex(columns=range(1, 11), fill_value=0).astype(float)
            Ib = b.assign(i=b['inning'].clip(1, 10)).pivot_table(
                index='pitcher_trackman_id', columns='i', aggfunc='size', fill_value=0
                ).reindex(columns=range(1, 11), fill_value=0).astype(float)
            cols = sorted(set(Pa.columns) & set(Pb.columns))
            ma = mix_a.reindex(Pa.index).reindex(columns=MIX).fillna(0.34).values
            mb = mix_b.reindex(Pb.index).reindex(columns=MIX).fillna(0.34).values
            ta, tb = Pa.values.sum(1), Pb.values.sum(1)
            c_sched = 1 - _mp_unit(Pa[cols].values) @ _mp_unit(Pb[cols].values).T
            c_inn = ((_mp_unit(Ia.values)[:, None, :] -
                      _mp_unit(Ib.values)[None, :, :]) ** 2).sum(-1)
            c_mix = ((ma[:, None, :] - mb[None, :, :]) ** 2).sum(-1)
            c_tot = (np.log1p(ta)[:, None] - np.log1p(tb)[None, :]) ** 2 * 0.05
            ha = a.groupby('pitcher_id')['pitcher_hand'].first().reindex(Pa.index).values
            hb = b.groupby('pitcher_trackman_id')['pitcher_hand'].first().reindex(Pb.index).values
            C = c_sched + c_inn + 2.0 * c_mix + c_tot + 100 * (ha[:, None] != hb[None, :])
            for i, j in zip(*linear_sum_assignment(C)):
                srt = np.sort(C[i])
                rows.append(dict(season=s, pitcher_id=Pa.index[i],
                                 pitcher_trackman_id=Pb.index[j], cost=C[i, j],
                                 margin=srt[1] - srt[0] if len(srt) > 1 else np.inf,
                                 n_tm=tb[j]))
    res = pd.DataFrame(rows)
    # 트레이드 선수는 여러 팀에서 후보가 나오므로 시즌별 1:1 로 정리
    best = res.sort_values('cost').groupby(['season', 'pitcher_id'], as_index=False).first()
    best = best.sort_values('cost').groupby(['season', 'pitcher_trackman_id'],
                                            as_index=False).first()
    vote = best.groupby(['pitcher_trackman_id', 'pitcher_id'])['n_tm'].sum().reset_index()
    win = (vote.sort_values('n_tm', ascending=False)
              .groupby('pitcher_trackman_id', as_index=False).first()
              .rename(columns={'pitcher_id': 'vote_pid'})[['pitcher_trackman_id', 'vote_pid']])
    best = best.merge(win, on='pitcher_trackman_id')
    # 검증은 반드시 다수결 '이전' 값으로. 교정 후에는 정의상 100%라 증거가 못 된다.
    g = best.groupby('pitcher_trackman_id')['pitcher_id']
    multi = g.nunique()[g.size() > 1]
    print(f"  [검증] 교정 전 시즌간 일관성 {(multi == 1).mean() * 100:.1f}% "
          f"(2시즌+ 등장 {len(multi)}명)")
    print(f"  [투수] 시즌간 다수결 교정 {int((best['pitcher_id'] != best['vote_pid']).sum())} "
          f"/ {len(best)}쌍")
    best['pitcher_id'] = best['vote_pid']
    out = best[['season', 'pitcher_id', 'pitcher_trackman_id', 'cost', 'margin']].copy()
    out['conf'] = np.where(out['cost'] <= out['cost'].quantile(0.75), 'high',
                    np.where(out['cost'] <= out['cost'].quantile(0.90), 'mid', 'low'))
    return out.sort_values(['season', 'pitcher_id']).reset_index(drop=True)
"""

exec(MAPPING_SRC)
print("build_pitcher_map 정의 완료")


In [ ]:
def build_rest_foul(tm):
    """등판 간 휴식 / 등판 밀도 / 파울 성향. 키와 컬럼 접두사를 feat_rp 와 맞춰
    저장·추론 경로를 그대로 재사용한다."""
    KEY = ['season', 'game_month', 'pitcher_id']
    t = tm.copy()
    t['_d'] = pd.to_datetime(t['game_date'], format='%m/%d/%Y', errors='coerce')
    out = (t.groupby(['pitcher_id', 'season', 'trackman_game_id'])
             .agg(_d=('_d', 'first'), n_pitch=('_d', 'size'),
                  game_month=('game_month', 'first')).reset_index()
             .sort_values(['pitcher_id', 'season', '_d']))
    out['rest'] = out.groupby(['pitcher_id', 'season'])['_d'].diff().dt.days
    mon = out.groupby(KEY).agg(
        rest_mean=('rest', 'mean'), rest_min=('rest', 'min'),
        b2b_rate=('rest', lambda s: float((s <= 1).mean()) if s.notna().any() else np.nan),
        n_out=('trackman_game_id', 'size'), pitch_per_out=('n_pitch', 'mean')).reset_index()
    t['_foul'] = t['pitch_of_pa'] - t['balls_before'] - t['strikes_before'] - 1
    fl = t.groupby(KEY).agg(foul_mean=('_foul', 'mean'),
                            pa_len=('pitch_of_pa', 'mean')).reset_index()
    mon = mon.merge(fl, on=KEY, how='outer')
    vals = ['rest_mean', 'rest_min', 'b2b_rate', 'n_out', 'pitch_per_out',
            'foul_mean', 'pa_len']
    # step17/18 과 동일한 leak-free 패턴: 그 달 '이전' 값만 쓴다
    mon = mon.sort_values(['pitcher_id', 'season', 'game_month'])
    g = mon.groupby('pitcher_id')
    for c in vals:
        mon['past_' + c] = g[c].transform(lambda s: s.shift(1).expanding().mean())
    return mon[KEY + ['past_' + c for c in vals]]


def step15_prep_trackman_data(trackman_df, pitcher_map_df):
    # 매핑에 season 이 있으면 반드시 시즌까지 키로 쓴다. pitcher_trackman_id 단독으로 붙이면
    # 한 투구가 여러 투수에게 중복 귀속돼 1.6배로 팽창한다 (2026-08-19 발견).
    keys = ['season', 'pitcher_trackman_id'] if 'season' in pitcher_map_df.columns \
        else ['pitcher_trackman_id']
    tm = pd.merge(trackman_df, pitcher_map_df[keys + ['pitcher_id']].drop_duplicates(),
                  on=keys, how='inner')
    if len(tm) > len(trackman_df):
        raise RuntimeError(f"트랙맨 병합이 팽창했습니다 ({len(trackman_df):,} -> {len(tm):,}). "
                           "매핑 키를 확인하세요.")
    b, s = tm['balls_before'], tm['strikes_before']
    p_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
    b_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
    neu = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
    tm['count_advantage'] = np.select([p_ahead, b_ahead, neu],
                                       ['Pitcher', 'Batter', 'Neutral'], default='None')
    tm['pitch_group'] = tm['pitch_type_group'].astype(str).str.lower()
    return tm[tm['pitch_group'].isin(['fastball', 'breaking', 'offspeed'])].copy()


def step16_calc_expected_difficulty(tm):
    groups = ['fastball', 'breaking', 'offspeed']
    sit = tm.groupby(['season', 'game_month', 'pitcher_id', 'count_advantage', 'pitch_group']
                     ).size().unstack(fill_value=0).reset_index()
    for c in groups:
        if c not in sit.columns:
            sit[c] = 0
    sit = sit.sort_values(by=['pitcher_id', 'count_advantage', 'season', 'game_month'])
    g = sit.groupby(['pitcher_id', 'count_advantage'])
    sit['past_fb'] = g['fastball'].cumsum() - sit['fastball']
    sit['past_br'] = g['breaking'].cumsum() - sit['breaking']
    sit['past_off'] = g['offspeed'].cumsum() - sit['offspeed']
    tot = sit['past_fb'] + sit['past_br'] + sit['past_off']
    sit['past_total'] = tot
    sit['exp_fb_prob'] = np.where(tot > 0, sit['past_fb'] / tot, 0)
    sit['exp_br_prob'] = np.where(tot > 0, sit['past_br'] / tot, 0)
    sit['exp_off_prob'] = np.where(tot > 0, sit['past_off'] / tot, 0)

    dm = tm.groupby(['season', 'game_month', 'pitcher_id', 'pitch_group'])[['rel_height', 'rel_side']].std()
    dm['diff_score'] = dm['rel_height'] + dm['rel_side']
    dm = dm.reset_index()
    dp = dm.pivot_table(index=['season', 'game_month', 'pitcher_id'],
                        columns='pitch_group', values='diff_score', fill_value=np.nan).reset_index()
    for c in groups:
        if c not in dp.columns:
            dp[c] = 0
    dp = dp.sort_values(by=['pitcher_id', 'season', 'game_month'])
    gd = dp.groupby(['pitcher_id'])
    dp['past_fb_diff'] = gd['fastball'].transform(lambda x: x.shift(1).expanding().mean())
    dp['past_br_diff'] = gd['breaking'].transform(lambda x: x.shift(1).expanding().mean())
    dp['past_off_diff'] = gd['offspeed'].transform(lambda x: x.shift(1).expanding().mean())

    res = pd.merge(sit, dp, on=['season', 'game_month', 'pitcher_id'], how='left')
    res['expected_control_difficulty'] = (res['exp_fb_prob'] * res['past_fb_diff']
                                          + res['exp_br_prob'] * res['past_br_diff']
                                          + res['exp_off_prob'] * res['past_off_diff'])
    return res[['season', 'game_month', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']]


def step17_calc_pitch_speed(tm):
    fb = tm[tm['pitch_group'] == 'fastball']
    sp = fb.groupby(['season', 'game_month', 'pitcher_id'])['rel_speed'].mean().reset_index()
    sp = sp.sort_values(by=['pitcher_id', 'season', 'game_month'])
    sp['past_fb_speed_mean'] = sp.groupby(['pitcher_id'])['rel_speed'].transform(
        lambda x: x.shift(1).expanding().mean())
    return sp[['season', 'game_month', 'pitcher_id', 'past_fb_speed_mean']]


def step18_calc_pitch_consistency_by_group(tm):
    groups = ['fastball', 'breaking', 'offspeed']
    metrics = ['rel_height_std', 'rel_side_std', 'extension_std',
               'spin_rate_std', 'vert_break_std', 'horz_break_std']
    cm = tm.groupby(['season', 'game_month', 'pitcher_id', 'pitch_group']).agg(
        rel_height_std=('rel_height', 'std'), rel_side_std=('rel_side', 'std'),
        extension_std=('extension', 'std'), spin_rate_std=('spin_rate', 'std'),
        vert_break_std=('induced_vert_break', 'std'), horz_break_std=('horz_break', 'std')
    ).reset_index()
    pv = cm.pivot_table(index=['season', 'game_month', 'pitcher_id'],
                        columns='pitch_group', values=metrics, fill_value=np.nan)
    pv.columns = [f"{grp}_{val}" for val, grp in pv.columns]
    pv = pv.reset_index()
    for pg in groups:
        for m in metrics:
            if f"{pg}_{m}" not in pv.columns:
                pv[f"{pg}_{m}"] = np.nan
    pv = pv.sort_values(by=['pitcher_id', 'season', 'game_month'])
    g = pv.groupby(['pitcher_id'])
    out_cols = ['season', 'game_month', 'pitcher_id']
    for pg in groups:
        for m in metrics:
            src, dst = f"{pg}_{m}", f"past_{pg}_{m}"
            pv[dst] = g[src].transform(lambda x: x.shift(1).expanding().mean())
            out_cols.append(dst)
    return pv[out_cols]


# ================= 릴리스 동역학 (2026-08-20 추가) =================
# 기존 step16~18 은 전부 (투수 x 월) 단위 표준편차라 세 가지가 한 숫자로 뭉개진다:
#   (a) 투구 간 기계적 흔들림  (b) 등판 간 드리프트  (c) 상황별 의도적 변화
# trackman_game_id / pitch_no 를 쓰면 분리할 수 있는데 여태 안 쓰고 있었다.
# 실제로 within(0.0304) 과 between(0.0274) 이 비슷한 크기 -> 절반이 다른 성분이었다.
# 검증: 2024 홀드아웃 6-seed 짝지어 +20(원본)/+22(재중심화).
#       절대점수 기준 신규 최저 736 > 기준선 평균 728 (기준선이 흔들려 차이 편차가 큼).

REL = ['rel_height', 'rel_side']


def build_release_dynamics(tm):
    """tm: step15 를 통과한 트랙맨 (pitcher_id 부착, 구종군 필터됨)"""
    t = tm.sort_values(['pitcher_id', 'trackman_game_id', 'pitch_no']).copy()
    out_key = ['season', 'game_month', 'pitcher_id', 'trackman_game_id']

    # --- 1) 연속 투구 간 릴리스 이동량 (같은 등판, 같은 구종군) ---
    g = t.groupby(out_key + ['pitch_group'], sort=False)
    t['seq_jump'] = np.sqrt(g['rel_height'].diff() ** 2 + g['rel_side'].diff() ** 2)

    # --- 2) 등판 단위 집계 ---
    agg = {'seq_jump': ('seq_jump', 'mean'), 'n': ('rel_height', 'size')}
    for c in REL + ['extension']:
        agg[f'w_{c}'] = (c, 'std')      # 등판 내 흔들림
        agg[f'm_{c}'] = (c, 'mean')     # 등판 중심 (등판 간 드리프트 계산용)
    outing = t.groupby(out_key, sort=False).agg(**agg).reset_index()
    outing = outing[outing.n >= 5]      # 5구 미만 등판은 통계가 무의미

    # --- 3) 등판 내 구속 감소 (fastball) ---
    fb = t[t.pitch_group == 'fastball'].copy()
    fb['rk'] = fb.groupby(out_key, sort=False).cumcount()
    fb['tot'] = fb.groupby(out_key, sort=False)['rk'].transform('size')
    fb = fb[fb.tot >= 9]
    fb['part'] = np.where(fb.rk < fb.tot / 3, 'early',
                   np.where(fb.rk >= 2 * fb.tot / 3, 'late', 'mid'))
    sp = fb[fb.part != 'mid'].pivot_table(index=out_key, columns='part',
                                          values='rel_speed', aggfunc='mean')
    sp['fb_speed_decay'] = sp.get('late', np.nan) - sp.get('early', np.nan)
    outing = outing.merge(sp[['fb_speed_decay']].reset_index(), on=out_key, how='left')

    # --- 4) 월 단위로 모으기: within 은 평균, between 은 등판중심의 표준편차 ---
    mkey = ['season', 'game_month', 'pitcher_id']
    m = outing.groupby(mkey).agg(
        seq_jump=('seq_jump', 'mean'),
        within_rel_h=('w_rel_height', 'mean'), within_rel_s=('w_rel_side', 'mean'),
        within_ext=('w_extension', 'mean'),
        between_rel_h=('m_rel_height', 'std'), between_rel_s=('m_rel_side', 'std'),
        between_ext=('m_extension', 'std'),
        fb_speed_decay=('fb_speed_decay', 'mean'),
        n_outing=('n', 'size'),
    ).reset_index()

    # --- 5) 터널링: 구종군 간 릴리스 중심 거리 ---
    cen = t.groupby(mkey + ['pitch_group'])[REL].mean().unstack('pitch_group')
    def gap(a, b):
        try:
            return np.sqrt((cen[('rel_height', a)] - cen[('rel_height', b)]) ** 2
                           + (cen[('rel_side', a)] - cen[('rel_side', b)]) ** 2)
        except KeyError:
            return pd.Series(np.nan, index=cen.index)
    tun = pd.DataFrame({'tunnel_fb_br': gap('fastball', 'breaking'),
                        'tunnel_fb_off': gap('fastball', 'offspeed')}).reset_index()
    m = m.merge(tun, on=mkey, how='left')

    # --- 6) 카운트 압박 하 릴리스 흔들림 차 ---
    cs = t.groupby(mkey + ['count_advantage'])[REL].std()
    cs = (cs['rel_height'] + cs['rel_side']).unstack('count_advantage')
    if 'Batter' in cs.columns and 'Pitcher' in cs.columns:
        m = m.merge((cs['Batter'] - cs['Pitcher']).rename('cnt_rel_gap').reset_index(),
                    on=mkey, how='left')
    else:
        m['cnt_rel_gap'] = np.nan

    # --- 7) leak-free 누적: 그 달 이전까지의 평균 ---
    cols = [c for c in m.columns if c not in mkey]
    m = m.sort_values(['pitcher_id', 'season', 'game_month'])
    gp = m.groupby('pitcher_id')
    for c in cols:
        m['past_' + c] = gp[c].transform(lambda x: x.shift(1).expanding().mean())
    return m[mkey + ['past_' + c for c in cols]]


# ================= 조건부 투수통계 (2026-08-18 추가) =================
# 설계: 성공률이 매 시즌 단조 하락(.565->.486)하므로 원시 성공률을 그대로 쓰면 과거 시즌의
#       높은 수준이 그대로 섞여 들어온다. 그래서 '그 시즌 리그평균 대비 편차'로 디트렌드한 뒤
#       0(=리그평균)으로 shrink 하는 경험적 베이즈 방식을 쓴다.
#       표본이 적은 조합일수록 자동으로 0에 가까워지므로 콜드스타트도 자연히 처리된다.
# 검증: 2024 홀드아웃 3-seed 짝지어 비교에서 기준선 대비 +20(원본)/+27(재중심화)
COND_SPECS = [
    (['pitcher_id'],                                    200, 'cond_p'),
    (['pitcher_id', 'count_advantage'],                 100, 'cond_pc'),
    (['pitcher_id', 'batter_hand'],                     100, 'cond_ph'),
    (['pitcher_id', 'batter_hand', 'count_advantage'],   50, 'cond_phc'),
]
if USE_COND_PB:
    COND_SPECS.append((['pitcher_id', 'batter_id'], 20, 'cond_pb'))


def _add_dev(df):
    """control_success 를 '그 시즌 리그평균 대비 편차'로 변환 (드리프트 제거)."""
    lg = df.groupby('season')['control_success'].mean()
    return df['control_success'] - df['season'].map(lg)


def build_cond_table(src, keys, C, name, target_season):
    # 시즌 감쇠. COND_DECAY = 1.0 이면 w 가 전부 1 이라 원래 식(sum/(count+C))과 완전히 같다.
    #
    # 음수는 '정규화 끔'. 2026-08-21 리더보드에서 정규화판(0.25)이 -11.05 로 최대 범인이었다.
    # 원인: w 합을 행 수에 맞추면 분모가 6시즌치(1800+C)인데 분자는 사실상 최근 1시즌치라
    # 과신이 된다. 정규화를 빼면 분모가 유효표본(400+C)으로 줄어 자동으로 더 shrink 된다 —
    # 표본이 작으면 0(리그평균)에 붙는 원래 설계 의도가 그대로 살아난다.
    _d = abs(COND_DECAY)
    w = _d ** ((target_season - 1) - src['season'].to_numpy())
    if COND_DECAY > 0:
        w = w * (len(src) / w.sum())
    t = src[keys].copy()
    t['_w'] = w
    t['_wd'] = w * src['_dev'].to_numpy()
    g = t.groupby(keys, observed=True)[['_wd', '_w']].sum().reset_index()
    g[name] = g['_wd'] / (g['_w'] + C)             # 0(리그평균)으로 shrink
    return g[keys + [name]]


def attach_cond_features(df):
    """학습용: 각 행은 '그 시즌보다 과거' 데이터로만 인코딩 -> leak-free.
    (배포 시 2025 test 가 2019~2024 로 인코딩되는 것과 동일한 규칙)"""
    df = df.copy()
    df['_dev'] = _add_dev(df)
    seasons = sorted(df['season'].unique())
    for keys, C, name in COND_SPECS:
        col = np.full(len(df), np.nan)
        for s in seasons:
            past = df[df['season'] < s]
            if len(past) == 0:
                continue
            t = build_cond_table(past, keys, C, name, s).set_index(keys)[name]
            cur = (df['season'] == s).values
            sl = df.loc[cur, keys]
            idx = pd.MultiIndex.from_frame(sl) if len(keys) > 1 else pd.Index(sl[keys[0]])
            col[cur] = t.reindex(idx).values
        df[name] = col
        print(f"  {name}: 결측 {np.isnan(col).mean()*100:.1f}% (첫 시즌 + 신규투수)")
    return df.drop(columns=['_dev'])


def build_all_cond_tables(df):
    """추론용: 학습 전 시즌을 다 써서 만든 최종 룩업 테이블."""
    d = df.copy()
    d['_dev'] = _add_dev(d)
    _ts = int(d['season'].max()) + 1        # 추론 대상 시즌(2025)이 감쇠 기준점
    out = {name: build_cond_table(d, keys, C, name, _ts) for keys, C, name in COND_SPECS}
    for _n, _t in out.items():
        print(f"  룩업 {_n}: {len(_t):,}행")
    return out


COND_COLS = [name for _, _, name in COND_SPECS]


# ======== 당해 시즌 성적 복원 (2026-08-22 EDA) ========
# asof_pitcher_*_rate 는 커리어 누적이라 투수마다 섞인 시즌 수가 다르다.
# 이력 있는 투수에게는 2019(리그 .5647)부터 섞인 낡은 값이고 신규 투수에게는
# 순수한 당해 시즌 값이다 — 2024 단독 예측 스킬이 141 vs 906 으로 갈린다.
# 그 시즌 시작 시점의 커리어 누적을 빼면 모든 투수에게 깨끗한 당해 시즌 값이 남는다.
WS_RATES = ['success', 'middle', 'reverse', 'ball', 'strike']
WS_C = 100.0                      # 표본이 적을 때 리그평균으로 shrink
WS_COLS = ['w_n', 'w_share'] + ['w_' + _k for _k in WS_RATES]


def _ws_rate_cols():
    return ['asof_pitcher_%s_rate' % _k for _k in WS_RATES]


def ws_season_means(df):
    """시즌별 리그평균 (각 비율마다). 디트렌드 기준이 된다."""
    return {c: df.groupby('season')[c].mean().to_dict() for c in _ws_rate_cols()}


def ws_next_season_mean(means, target_season):
    """대상 시즌의 리그평균을 학습 시즌만으로 외삽 (최근 3시즌 선형).
    2024 를 이 방식으로 맞히면 오차 0.0016 이다 (CLAUDE.md 1장)."""
    out = {}
    for c, d in means.items():
        ss = sorted(d)
        k = ss[-3:]
        a, b = np.polyfit(k, [d[s] for s in k], 1)
        out[c] = float(np.clip(a * target_season + b, 0.0, 1.0))
    return out


def ws_apply(df, prior_n, prior_x, lg):
    """행별로 당해 시즌 값을 복원해 WS_COLS 를 붙인다.

    prior_n : Series/array — 그 행의 '대상 시즌 시작 시점' 커리어 투구수
    prior_x : dict[rate_col] -> array — 같은 시점의 누적 개수
    lg      : dict[rate_col] -> 그 행이 속한 시즌의 리그평균 (array 또는 scalar)
    """
    n = df['asof_pitcher_n'].to_numpy(dtype='float64')
    wn = np.maximum(n - np.asarray(prior_n, dtype='float64'), 0.0)
    df['w_n'] = wn
    df['w_share'] = wn / np.maximum(n, 1.0)
    for c, k in zip(_ws_rate_cols(), WS_RATES):
        x = (df[c].fillna(0).to_numpy(dtype='float64') * n).round()
        wx = np.clip(x - np.asarray(prior_x[c], dtype='float64'), 0.0, wn)
        base = np.asarray(lg[c], dtype='float64')
        df['w_' + k] = (wx + base * WS_C) / (wn + WS_C) - base
    return df




# ======== 타자측 당해 시즌 복원 (2026-08-23) ========
# 투수측(w_*)과 완전히 같은 수법. asof_batter_*_rate 도 커리어 누적이다.
WB_RATES = ['success', 'middle']
WB_C = 100.0
WB_COLS = ['wb_n', 'wb_share'] + ['wb_' + _k for _k in WB_RATES]


def _wb_rate_cols():
    return ['asof_batter_%s_rate' % _k for _k in WB_RATES]


def wb_season_means(df):
    return {c: df.groupby('season')[c].mean().to_dict() for c in _wb_rate_cols()}


def wb_apply(df, prior_n, prior_x, lg):
    n = df['asof_batter_n'].to_numpy(dtype='float64')
    wn = np.maximum(n - np.asarray(prior_n, dtype='float64'), 0.0)
    df['wb_n'] = wn
    df['wb_share'] = wn / np.maximum(n, 1.0)
    for c, k in zip(_wb_rate_cols(), WB_RATES):
        x = (df[c].fillna(0).to_numpy(dtype='float64') * n).round()
        wx = np.clip(x - np.asarray(prior_x[c], dtype='float64'), 0.0, wn)
        base = np.asarray(lg[c], dtype='float64')
        df['wb_' + k] = (wx + base * WB_C) / (wn + WB_C) - base
    return df




# ======== prev-game 시즌 경계 보정 (2026-08-23) ========
# prev5 에 작년분이 섞인 행이 22.6% 다. 그 구간에서 이 피처는 적극적인 독이다
# (당해 0~60구 & 이력있는 투수 82,826행에서 prev5 단독 스킬 -3,224).
PF_SPEC = [('asof_pitcher_prev1_game_success_rate', 1),
           ('asof_pitcher_prev3_game_success_rate', 3),
           ('asof_pitcher_prev5_game_success_rate', 5),
           ('asof_pitcher_prev1_game_middle_rate', 1),
           ('asof_pitcher_prev3_game_middle_rate', 3),
           ('asof_pitcher_prev5_game_middle_rate', 5)]
PF_COLS = (['pf_gest'] + ['pf_cross%d' % w for w in (1, 3, 5)]
           + ['pf_' + c.replace('asof_pitcher_', '') for c, _ in PF_SPEC])


def pf_build_appearance(df):
    """투수별 **등판당 평균 투구수**. 학습·추론 양쪽에서 같은 함수를 쓴다.

    train 을 경기 단위로 복원해서 센다. (season, month, dow, game_type) 변화
    또는 이닝 하락에서 자르면 R 이 시즌마다 정확히 720경기가 나온다 (KBO 정규시즌).

    ⚠️⚠️ 복원은 **원본 행 순서**에 의존한다. run_full_pipeline 은 트랙맨 merge_asof
    때문에 time_idx 로 정렬하므로(claude.md 4-15) 파이프라인 안에서 부르면 깨진다 —
    실제로 첫 빌드에서 경기가 4,868개가 아니라 213,726개로 쪼개졌고 등판당 투구수가
    24.9 대신 3.1 이 나왔다. 학습은 cross≈0, 추론은 cross=22.6% 로 **규칙이 갈렸다**
    (트랙맨 exact/asof -36점과 같은 실패 유형).
    row_id 가 제로패딩(TRAIN_0000001)이라 정렬만으로 원본 순서가 복원된다.
    """
    d = df.sort_values('row_id') if 'row_id' in df.columns else df
    k = d[['season', 'game_month', 'game_dayofweek', 'game_type']].astype(str).agg(
        '|'.join, axis=1)
    gid = ((k != k.shift()) | (d['inning'].diff() < 0)).cumsum()
    ng = gid.nunique()
    ap = d.assign(_g=gid).groupby(['pitcher_id', '_g']).size()
    avg = ap.groupby(level=0).mean().rename('pf_avg_pa').reset_index()
    med = avg['pf_avg_pa'].median()
    print("  경기 복원 %d개 | 등판 %d개 | 등판당 투구수 중앙 %.1f" % (ng, len(ap), med))
    # 알려진 정답으로 자체 검증한다 — 6시즌 x (R 720 + F 76~104) = 4,868
    if not (4300 <= ng <= 5400):
        raise RuntimeError("경기 복원이 %d개다 (기대 ~4,868). 행 순서가 원본이 아니다." % ng)
    if not (18.0 <= med <= 32.0):
        raise RuntimeError("등판당 투구수 중앙이 %.1f 다 (기대 ~24.9)." % med)
    return avg


def pf_season_means(df):
    return {c: df.groupby('season')[c].mean().to_dict() for c, _ in PF_SPEC}


def pf_apply(df, avg_pa, lg_cur, lg_prv):
    """행별로 prev-game 지표를 보정한다.

    avg_pa : array — 그 투수의 등판당 평균 투구수
    lg_cur : dict[col] -> array/scalar — 그 행이 속한 시즌의 리그평균
    lg_prv : dict[col] -> array/scalar — 그 직전 시즌의 리그평균
    """
    wn = df['w_n'].to_numpy(dtype='float64')          # wseason 이 만들어둔 당해 투구수
    gest = wn / np.maximum(np.asarray(avg_pa, dtype='float64'), 1.0)
    df['pf_gest'] = gest
    seen = set()
    for c, w in PF_SPEC:
        cross = np.clip(w - gest, 0.0, w) / w
        if w not in seen:
            df['pf_cross%d' % w] = cross              # (B) 관련성: 작년분 비율
            seen.add(w)
        a = np.asarray(lg_cur[c], dtype='float64')
        b = np.asarray(lg_prv[c], dtype='float64')
        b = np.where(np.isfinite(b), b, a)
        blend = a * (1.0 - cross) + b * cross         # (A) 수준: 가리키는 시즌으로
        df['pf_' + c.replace('asof_pitcher_', '')] = \
            df[c].to_numpy(dtype='float64') - blend
    return df


def attach_prevfix(df):
    """학습용. attach_wseason 뒤에 불러야 한다 (w_n 이 필요하다)."""
    df = df.copy()
    means = pf_season_means(df)
    avg = pf_build_appearance(df).set_index('pitcher_id')['pf_avg_pa']
    a = df['pitcher_id'].map(avg).to_numpy(dtype='float64')
    a = np.where(np.isfinite(a), a, np.nanmedian(a))
    lg_cur = {c: df['season'].map(means[c]).to_numpy(dtype='float64')
              for c, _ in PF_SPEC}
    lg_prv = {c: df['season'].sub(1).map(means[c]).to_numpy(dtype='float64')
              for c, _ in PF_SPEC}
    df = pf_apply(df, a, lg_cur, lg_prv)
    print("  prev-game 보정: 작년분 섞인 행 "
          + " ".join("prev%d %.1f%%" % (w, 100.0 * (df['pf_cross%d' % w] > 0).mean())
                     for w in (1, 3, 5)))
    return df


def attach_wsbat(df):
    """학습용. 각 행을 '그 시즌 시작 시점' 기준으로 분해 (leak-free)."""
    df = df.copy()
    means = wb_season_means(df)
    order = np.argsort(df['asof_batter_n'].to_numpy(dtype='float64'), kind='stable')
    first = df.iloc[order].groupby(['batter_id', 'season'], sort=False).head(1)
    key = pd.MultiIndex.from_arrays([df['batter_id'], df['season']])
    fi = first.set_index(['batter_id', 'season'])
    n0 = fi['asof_batter_n'].reindex(key).to_numpy(dtype='float64')
    px = {c: (fi[c].fillna(0).reindex(key).to_numpy(dtype='float64') * n0).round()
          for c in _wb_rate_cols()}
    lg = {c: df['season'].map(means[c]).to_numpy(dtype='float64')
          for c in _wb_rate_cols()}
    df = wb_apply(df, n0, px, lg)
    print("  타자 당해시즌 복원: 타석수 중앙값 %.0f | " % np.nanmedian(df['wb_n'])
          + " ".join("%s=%+.4f" % (k, np.nanmean(df['wb_' + k])) for k in WB_RATES))
    return df


def build_batter_prior(df):
    """추론용 룩업. 각 타자의 마지막 학습 시즌 끝 시점 커리어 상태."""
    order = np.argsort(df['asof_batter_n'].to_numpy(dtype='float64'), kind='stable')
    last = df.iloc[order].groupby('batter_id', sort=False).tail(1)
    n = last['asof_batter_n'].to_numpy(dtype='float64')
    out = pd.DataFrame({'batter_id': last['batter_id'].to_numpy(),
                        'wb_n0': n + 1.0})
    for c in _wb_rate_cols():
        out['wb_x0__' + c] = (last[c].fillna(0).to_numpy(dtype='float64') * n).round()
    return out.reset_index(drop=True)


def attach_wseason(df):
    """학습용. 각 행을 '그 시즌 시작 시점' 기준으로 분해한다 (leak-free)."""
    df = df.copy()
    means = ws_season_means(df)
    order = np.argsort(df['asof_pitcher_n'].to_numpy(dtype='float64'), kind='stable')
    first = df.iloc[order].groupby(['pitcher_id', 'season'], sort=False).head(1)
    key = pd.MultiIndex.from_arrays([df['pitcher_id'], df['season']])
    fi = first.set_index(['pitcher_id', 'season'])
    n0 = fi['asof_pitcher_n'].reindex(key).to_numpy(dtype='float64')
    px = {c: (fi[c].fillna(0).reindex(key).to_numpy(dtype='float64') * n0).round()
          for c in _ws_rate_cols()}
    lg = {c: df['season'].map(means[c]).to_numpy(dtype='float64') for c in _ws_rate_cols()}
    df = ws_apply(df, n0, px, lg)
    print("  당해시즌 복원: 투구수 중앙값 %.0f | " % np.nanmedian(df['w_n'])
          + " ".join("%s=%+.4f" % (k, np.nanmean(df['w_' + k])) for k in WS_RATES))
    return df


def build_pitcher_prior(df):
    """추론용 룩업. 각 투수의 **마지막 학습 시즌 끝** 시점 커리어 상태.
    test 행의 asof 에서 이걸 빼면 대상 시즌(2025) 값이 남는다."""
    order = np.argsort(df['asof_pitcher_n'].to_numpy(dtype='float64'), kind='stable')
    last = df.iloc[order].groupby('pitcher_id', sort=False).tail(1)
    n = last['asof_pitcher_n'].to_numpy(dtype='float64')
    out = pd.DataFrame({'pitcher_id': last['pitcher_id'].to_numpy(),
                        'w_n0': n + 1.0})       # 그 마지막 투구 자신도 포함
    for c in _ws_rate_cols():
        out['w_x0__' + c] = (last[c].fillna(0).to_numpy(dtype='float64') * n).round()
    return out.reset_index(drop=True)



## [Cell 3] 통합 파이프라인

In [ ]:
def run_full_pipeline(train_df, trackman_df, pitcher_map, trackman_mode='asof'):
    print(f"파이프라인 시작 (trackman_mode={trackman_mode})...")
    df_proc = train_df.copy()

    df_proc = step1_basic_features(df_proc)
    df_proc = step2_pitcher_role_features(df_proc)
    df_proc = step3_matchup_features(df_proc)
    df_proc = step4_refined_count_features(df_proc)
    df_proc = step5_pitches_per_inning(df_proc)
    df_proc = step6_combined_runner_features(df_proc)

    prior_mean = float(df_proc['asof_pitcher_success_rate'].mean())
    print(f"  prior_mean = {prior_mean:.6f}")

    df_proc = step7_bayesian_smoothing(df_proc, prior_mean=prior_mean)
    df_proc = step8_batter_toughness_features(df_proc)
    df_proc = step9_garbage_time_features(df_proc)
    df_proc = step10_recent_form_momentum(df_proc)
    df_proc = step11_veteran_and_pressure_features(df_proc)
    df_proc = step12_first_pitch_tendency(df_proc)
    df_proc = step13_sac_fly_threat(df_proc)

    if 'count_advantage' not in df_proc.columns:
        b, s = df_proc['balls_before'], df_proc['strikes_before']
        p_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
        b_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
        neu = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
        df_proc['count_advantage'] = np.select([p_ahead, b_ahead, neu],
                                                ['Pitcher', 'Batter', 'Neutral'], default='None')

    tm_base = step15_prep_trackman_data(trackman_df, pitcher_map)
    feat_diff = step16_calc_expected_difficulty(tm_base)
    feat_speed = step17_calc_pitch_speed(tm_base)
    feat_rp = step18_calc_pitch_consistency_by_group(tm_base)
    # 릴리스 동역학: 키가 feat_rp 와 같고 컬럼이 past_ 로 시작하므로 여기 합치면
    # 저장/추론/zip 로직이 수정 없이 그대로 따라온다.
    if USE_RELEASE_DYNAMICS:
        _dyn = build_release_dynamics(tm_base)
        _n0 = len(feat_rp)
        feat_rp = feat_rp.merge(_dyn, on=['season', 'game_month', 'pitcher_id'], how='outer')
        print(f"  릴리스 동역학 {len(_dyn):,}행 -> feat_rp {_n0:,} -> {len(feat_rp):,}행 "
              f"(신규 {len([c for c in _dyn.columns if c.startswith('past_')])}개)")
    if USE_REST_FOUL:
        _rf = build_rest_foul(tm_base)
        _n0 = len(feat_rp)
        feat_rp = feat_rp.merge(_rf, on=['season', 'game_month', 'pitcher_id'], how='outer')
        print(f"  휴식·파울 {len(_rf):,}행 -> feat_rp {_n0:,} -> {len(feat_rp):,}행 "
              f"(신규 {len([c for c in _rf.columns if c.startswith('past_')])}개)")
    rp_value_cols = [c for c in feat_rp.columns if c.startswith('past_')]

    if trackman_mode == 'asof':
        for f in [feat_diff, feat_speed, feat_rp]:
            f['time_idx'] = f['season'] * 100 + f['game_month']
            f.sort_values('time_idx', inplace=True)
        df_proc['time_idx'] = df_proc['season'] * 100 + df_proc['game_month']
        df_proc = df_proc.sort_values('time_idx')

        df_proc = pd.merge_asof(
            df_proc,
            feat_diff[['time_idx', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']],
            on='time_idx', by=['pitcher_id', 'count_advantage'], direction='backward')
        df_proc = pd.merge_asof(
            df_proc, feat_speed[['time_idx', 'pitcher_id', 'past_fb_speed_mean']],
            on='time_idx', by='pitcher_id', direction='backward')
        df_proc = pd.merge_asof(
            df_proc, feat_rp[['time_idx', 'pitcher_id'] + rp_value_cols],
            on='time_idx', by='pitcher_id', direction='backward')
        df_proc = df_proc.drop(columns=['time_idx'])
    else:
        df_proc = pd.merge(df_proc, feat_diff,
                           on=['season', 'game_month', 'pitcher_id', 'count_advantage'], how='left')
        df_proc = pd.merge(df_proc, feat_speed,
                           on=['season', 'game_month', 'pitcher_id'], how='left')
        df_proc = pd.merge(df_proc, feat_rp,
                           on=['season', 'game_month', 'pitcher_id'], how='left')
        for c in ['expected_control_difficulty', 'past_fb_speed_mean'] + rp_value_cols:
            if c in df_proc.columns:
                df_proc[c] = df_proc[c].fillna(0)

    # 조건부 투수통계 (step14 이전에 붙여야 함: pitcher_id/count_advantage 가 아직 원시 dtype)
    cond_tables = {}
    if USE_COND_STATS:
        print("조건부 투수통계 생성...")
        df_proc = attach_cond_features(df_proc)
        df_proc = attach_wseason(df_proc)
        df_proc = attach_wsbat(df_proc)
        df_proc = attach_prevfix(df_proc)
        cond_tables = build_all_cond_tables(df_proc)   # 추론용 최종 테이블(전 시즌)

    df_proc = step14_convert_to_category(df_proc)
    print("파이프라인 완료.")
    return df_proc.reset_index(drop=True), prior_mean, feat_diff, feat_speed, feat_rp, cond_tables


## [Cell 4] 데이터 로드 및 실행

In [ ]:
import glob as _g, os as _o
_PATTERNS = [
    "/kaggle/input/**/train.csv",
    "/content/drive/MyDrive/*/train.csv",
    "/content/drive/MyDrive/*/*/train.csv",
    "/content/drive/MyDrive/*/*/*/train.csv",
    "/content/*/train.csv",
    "./data/train.csv",
    "../data/train.csv",
]
DATA_DIR = None
for _p in _PATTERNS:
    for _c in sorted(_g.glob(_p, recursive=("**" in _p))):
        if _o.path.exists(_o.path.join(_o.path.dirname(_c), "trackman_history.csv")):
            DATA_DIR = _o.path.dirname(_c)
            break
    if DATA_DIR:
        break
if DATA_DIR is None:
    raise RuntimeError("train.csv + trackman_history.csv 를 못 찾음: " + str(_PATTERNS))
print("DATA_DIR =", DATA_DIR, flush=True)

df_train = pd.read_csv(f"{DATA_DIR}/train.csv")
df_trackman = pd.read_csv(f"{DATA_DIR}/trackman_history.csv")
print("train:", df_train.shape, "| trackman:", df_trackman.shape)

# 주최측이 준 pitcher_id_mapping.csv 는 약 91%가 틀렸다(2장 참고). 매번 다시 만든다.
print("투수 매핑 재구축...")
pitcher_id_mapping = build_pitcher_map(df_train, df_trackman)
print(f"  매핑 {len(pitcher_id_mapping)}행 | 2024 투구 커버리지 "
      f"{df_train[df_train.season == 2024].pitcher_id.isin(pitcher_id_mapping[pitcher_id_mapping.season == 2024].pitcher_id).mean() * 100:.1f}%")


In [ ]:
df_processed, PRIOR_MEAN, feat_diff, feat_speed, feat_rp, cond_tables = run_full_pipeline(
    df_train, df_trackman, pitcher_id_mapping, trackman_mode=TRACKMAN_MODE)
print("df_processed:", df_processed.shape)


## [Cell 5] 추론용 아티팩트 저장

In [ ]:
os.makedirs("model", exist_ok=True)

with open("model/train_constants.json", "w") as f:
    json.dump({"prior_mean": PRIOR_MEAN, "trackman_mode": TRACKMAN_MODE}, f)
print(f"train_constants.json  prior_mean={PRIOR_MEAN:.6f}  trackman_mode={TRACKMAN_MODE}")

# 트랙맨 테이블은 step16~18 출력 그대로 저장 (dropna/dedup 금지)
_rp = [c for c in feat_rp.columns if c.startswith('past_')]
_diff_cols = ['season', 'game_month', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']
_speed_cols = ['season', 'game_month', 'pitcher_id', 'past_fb_speed_mean']
_rp_cols = ['season', 'game_month', 'pitcher_id'] + _rp
if TRACKMAN_MODE == 'asof':
    for f_, extra in [(feat_diff, _diff_cols), (feat_speed, _speed_cols), (feat_rp, _rp_cols)]:
        if 'time_idx' not in f_.columns:
            f_['time_idx'] = f_['season'] * 100 + f_['game_month']
    _diff_cols = ['time_idx'] + _diff_cols
    _speed_cols = ['time_idx'] + _speed_cols
    _rp_cols = ['time_idx'] + _rp_cols

feat_diff[_diff_cols].to_csv("model/feat_diff.csv", index=False)
feat_speed[_speed_cols].to_csv("model/feat_speed.csv", index=False)
feat_rp[_rp_cols].to_csv("model/feat_rp.csv", index=False)
print(f"feat_diff {len(feat_diff):,} / feat_speed {len(feat_speed):,} / feat_rp {len(feat_rp):,}")

# 'None' 라운드트립 검증: 0-0/3-2 카운트를 뜻하는 실제 문자열인데
# pd.read_csv 기본 설정은 NaN으로 읽어버려 merge가 전량 실패한다.
_NA = ['', 'NaN', 'nan', 'NULL', 'null', 'NA', 'N/A', 'n/a']
_chk = pd.read_csv("model/feat_diff.csv", keep_default_na=False, na_values=_NA)
_n_none = (_chk['count_advantage'].astype(str) == 'None').sum()
_bad = pd.read_csv("model/feat_diff.csv")['count_advantage'].isna().sum()
print(f"\n'None' 행 {_n_none:,}개 — 기본 read_csv로는 {_bad:,}개가 NaN이 됨 (script.py는 na_values 명시)")
assert _n_none > 0, "'None' 값이 사라졌습니다"

# 조건부 투수통계 테이블 저장 (추론에서 룩업)
# count_advantage 의 'None' 은 0-0/3-2 를 뜻하는 실제 문자열이라 라운드트립 검증 필수 (4-3)
for _name, _tbl in cond_tables.items():
    _tbl.to_csv(f"model/{_name}.csv", index=False)
    print(f"{_name}.csv  {len(_tbl):,}행")
if 'cond_phc' in cond_tables:
    _c = pd.read_csv("model/cond_phc.csv", keep_default_na=False, na_values=_NA)
    assert (_c['count_advantage'].astype(str) == 'None').sum() > 0, "'None' 유실"
    print("조건부 테이블 'None' 라운드트립 OK")

# ---- 당해 시즌 복원용 룩업 + 대상 시즌 리그평균 외삽 ----
_pp = build_pitcher_prior(df_train)
_pp.to_csv("model/pitcher_prior.csv", index=False)
_tgt = int(df_train['season'].max()) + 1
_ws_lg = ws_next_season_mean(ws_season_means(df_train), _tgt)
with open("model/train_constants.json", "r") as f:
    _tc = json.load(f)
_tc["ws_target_season"] = _tgt
_tc["ws_league_mean"] = _ws_lg
_tc["ws_rates"] = WS_RATES
_tc["ws_C"] = WS_C
with open("model/train_constants.json", "w") as f:
    json.dump(_tc, f)
print(f"pitcher_prior.csv  {len(_pp):,}행 | {_tgt} 리그평균 외삽 "
      + " ".join(f"{k.split('_')[-2]}={v:.4f}" for k, v in _ws_lg.items()))

# ---- 타자측 당해 시즌 복원용 룩업 ----
_bp = build_batter_prior(df_train)
_bp.to_csv("model/batter_prior.csv", index=False)
_wb_lg = ws_next_season_mean(wb_season_means(df_train), _tgt)
with open("model/train_constants.json", "r") as f:
    _tc = json.load(f)
_tc["wb_league_mean"] = _wb_lg
_tc["wb_rates"] = WB_RATES
_tc["wb_C"] = WB_C
with open("model/train_constants.json", "w") as f:
    json.dump(_tc, f)
print(f"batter_prior.csv  {len(_bp):,}행 | "
      + " ".join(f"{k}={v:.4f}" for k, v in _wb_lg.items()))

# ---- prev-game 보정용 룩업 + 대상 시즌 리그평균 외삽 ----
_pf = pf_build_appearance(df_train)
_pf.to_csv("model/pitcher_appearance.csv", index=False)
_pf_means = pf_season_means(df_train)
_pf_cur = ws_next_season_mean(_pf_means, _tgt)                 # 2025 외삽
_pf_prv = {c: float(_pf_means[c][max(_pf_means[c])]) for c, _ in PF_SPEC}  # 2024 실측
with open("model/train_constants.json", "r") as f:
    _tc = json.load(f)
_tc["pf_league_cur"] = _pf_cur
_tc["pf_league_prev"] = _pf_prv
_tc["pf_spec"] = [[c, w] for c, w in PF_SPEC]
with open("model/train_constants.json", "w") as f:
    json.dump(_tc, f)
print(f"pitcher_appearance.csv  {len(_pf):,}행 | {_tgt} 리그평균 외삽 완료")


## [Cell 6a] Optuna 하이퍼파라미터 탐색 (안전, 로컬 판단 가능)

**이건 리더보드 도박이 아니다.** `StratifiedKFold` 자체는 "미래 시즌 예측력"을 정직하게
재지 못하지만, "이 하이퍼파라미터가 주어진 학습 데이터를 얼마나 잘 맞히는가"(OOF Brier)는
`StratifiedKFold`로도 정직하게 잴 수 있다. 그래서 튜닝은 로컬에서 안전하게 결정한다.

**속도를 위한 설계**: 매 trial마다 10-fold를 다 돌리면 40 trial × 10 fold라 너무 느리다.
대신 데이터의 30%만 표본으로 뽑아 3-fold로 빠르게 채점한다. 최종 학습(Cell 6b)은 이렇게
찾은 파라미터로 전체 데이터 × 10-fold × 3-seed를 학습한다.


In [ ]:
try:
    import optuna
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)
    import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import brier_score_loss
from catboost import CatBoostClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

target_col = 'control_success'
drop_cols = [target_col, 'row_id', 'pitcher_id', 'batter_id', 'time_idx']
drop_cols += DEAD_FEATURES   # Cell 0 에서 정의 (커리어누적 오해로 죽은 피처들)
drop_cols += DROP_CAL        # 절개 실험: 변형 m 에서만 비어있지 않다
feature_cols = [c for c in df_processed.columns if c not in drop_cols]

X_full = df_processed[feature_cols].copy()
y_full = df_processed[target_col].copy()
for col in [c for c in X_full.columns if X_full[c].dtype.name in ['category', 'object']]:
    X_full[col] = X_full[col].astype(str).astype('category')
cat_features = [c for c in X_full.columns if X_full[c].dtype.name == 'category']

with open("model/selected_features.json", "w") as f:
    json.dump(list(feature_cols), f)
print(f"피처 {len(feature_cols)}개 (범주형 {len(cat_features)}개)")

# 탐색용 30% 서브샘플 (계층 유지)
# skf.split()은 (train_idx, test_idx) 순서로 반환한다. 30%에 가까운 건 test_idx(약 33%) 쪽이므로
# 두 번째 원소를 받는다 (첫 번째를 받으면 train_idx=약 67%가 되어 의도보다 훨씬 커진다).
_, _sub_idx = next(StratifiedKFold(n_splits=3, shuffle=True, random_state=0).split(X_full, y_full))
X_sub, y_sub = X_full.iloc[_sub_idx], y_full.iloc[_sub_idx]
print(f"Optuna 탐색용 서브샘플: {len(X_sub):,}행 (전체의 약 {len(X_sub)/len(X_full):.0%})")


def objective(trial):
    params = {
        "iterations": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 0.5, 3.0),
        "eval_metric": "Logloss",
        "cat_features": cat_features,
        "random_seed": 42,
        "task_type": "GPU",
        "early_stopping_rounds": 50,
    }
    skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)
    briers = []
    for tr_idx, val_idx in skf3.split(X_sub, y_sub):
        model = CatBoostClassifier(**params)
        model.fit(X_sub.iloc[tr_idx], y_sub.iloc[tr_idx],
                  eval_set=(X_sub.iloc[val_idx], y_sub.iloc[val_idx]), verbose=0)
        p = model.predict_proba(X_sub.iloc[val_idx])[:, 1]
        briers.append(brier_score_loss(y_sub.iloc[val_idx], p))
    return float(np.mean(briers))


if RUN_OPTUNA:
    print("\n=== Optuna 탐색 시작 ===")
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    _found = dict(study.best_params)
    print(f"\n최적 Brier: {study.best_value:.5f}")
else:
    print("\n=== Optuna 생략 (RUN_OPTUNA=False) — v4 파라미터 재사용 ===")
    _found = dict(V4_BEST_PARAMS)

BEST_PARAMS = _found
BEST_PARAMS["iterations"] = 1000
BEST_PARAMS["eval_metric"] = "Logloss"
BEST_PARAMS["task_type"] = "GPU"
BEST_PARAMS["early_stopping_rounds"] = 50
BEST_PARAMS["cat_features"] = cat_features  # 누락돼 있었음 -- 없으면 Cell 6b의 .fit()에서 CatBoostError로 크래시함

print("최종 파라미터:", BEST_PARAMS)

with open("model/best_params.json", "w") as f:
    json.dump(BEST_PARAMS, f, indent=2)
# ================= aux_rev: 실패유형 보조 타겟 (교차적합) =================
AUX_ITERS = 300
AUX_FOLDS = 3


def _recover_reverse(df):
    """asof 인접 행 차분으로 **투구 단위** reverse 라벨을 복원한다 (train 전용).

    `asof_pitcher_n` 은 투수 내에서 정확히 +1 씩 증가하므로 인접 두 행이
    한 투구 차이다. asof 는 "직전까지" 이므로 행 i 와 i+1 로 투구 i 의 라벨이 나온다.
    ⚠️ (pitcher_id, asof_pitcher_n) 정렬이 전제다 — df_processed 는 time_idx 로
       정렬돼 있다 (claude.md 4-15). success 로 검산해 틀리면 즉시 중단한다.
    ⚠️ 규정: 차분은 train 에서만. test 인접 행 차분은 주최측이 "규칙 위반" 이라
       답한 사안이다 (2jin1 08-17). train 유래 값에는 제약이 없다 (DACON.GM 08-19).
    """
    n = df["asof_pitcher_n"].to_numpy(dtype="float64")
    g = df["pitcher_id"].to_numpy()
    o = np.lexsort((n, g))
    ns, gs = n[o], g[o]
    ok = np.r_[(gs[:-1] == gs[1:]) & (ns[1:] - ns[:-1] == 1), False]
    out = {}
    for k in ("reverse", "success"):
        cum = np.round(df["asof_pitcher_%s_rate" % k].fillna(0)
                       .to_numpy(dtype="float64")[o] * ns)
        d = np.r_[cum[1:] - cum[:-1], np.nan]
        v = np.where(ok & np.isin(d, [0.0, 1.0]), d, np.nan)
        b = np.full(len(df), np.nan)
        b[o] = v
        out[k] = b
    m = np.isfinite(out["success"])
    acc = float((out["success"][m] == df["control_success"].to_numpy()[m]).mean())
    print("  라벨 복원 검산: success 일치율 %.6f (표본 %s)"
          % (acc, format(int(m.sum()), ",")))
    if acc < 0.999:
        raise RuntimeError("복원 검산 실패 %.6f -- 행 순서를 의심할 것" % acc)
    return out["reverse"]


def _fit_aux(Xh, tgt, Xv, tag, params=None):
    """보조 모델을 교차적합한다. 학습행은 OOF, 예측행은 폴드 평균.

    ⚠️ reverse=1 이면 success=0 이므로 in-sample 적합은 타겟을 통째로 흘린다.
    """
    _p = dict(BEST_PARAMS if params is None else params)
    _p["iterations"] = AUX_ITERS
    _p.pop("early_stopping_rounds", None)
    _p["eval_metric"] = "Logloss"
    oof = np.full(len(Xh), np.nan)
    vp = np.zeros(len(Xv)) if Xv is not None else None
    fit = np.isfinite(tgt)
    ms = []
    _sk = StratifiedKFold(n_splits=AUX_FOLDS, shuffle=True, random_state=7)
    for _k, (_ti, _vi) in enumerate(_sk.split(Xh, np.nan_to_num(tgt, nan=0.0).astype(int))):
        _ti = _ti[fit[_ti]]
        _m = CatBoostClassifier(**_p)
        _m.fit(Xh.iloc[_ti], tgt[_ti].astype(int), verbose=0)
        oof[_vi] = _m.predict_proba(Xh.iloc[_vi])[:, 1]
        if Xv is not None:
            vp += _m.predict_proba(Xv)[:, 1] / AUX_FOLDS
        ms.append(_m)
        print("  %s aux fold %d/%d" % (tag, _k + 1, AUX_FOLDS))
    return oof, vp, ms


_rev = _recover_reverse(df_processed)
print("  reverse 율 %.4f | 결측 %.2f%%" % (np.nanmean(_rev), 100 * np.isnan(_rev).mean()))

# ⚠️ 보조 모델에서 season / game_type 을 **뺀다** (2026-08-25 리더보드 진단).
#    auxrev v1 은 스크리너 +40.2 인데 LB +4.14 (전달률 0.10) 였다. 원인은
#    보조 모델의 중요도 절반이 season(15.1%) x game_type(15.9%) = 드리프트였고,
#    season 경계가 2023.5 에서 끝나(eda31) 2024 수준에 고정된 값을 2025 에 뱉기
#    때문이다. 빼고 재니 2023 이 **-18.1 -> +55.0** 으로 뒤집혔다 (2024 +22.5).
#    본 모델은 season/game_type 을 직접 갖고 있으므로 잃는 것이 없다.
AUX_DROP = ["season", "game_type"]
_aux_cols = [c for c in feature_cols if c not in AUX_DROP]
_aux_params = dict(BEST_PARAMS)
_aux_params["cat_features"] = [c for c in cat_features if c not in AUX_DROP]
print("  보조 모델 피처 %d개 (제거: %s)" % (len(_aux_cols), AUX_DROP))

# 배포용: 전체 train 으로 교차적합. 모델 3개를 zip 에 실어 추론 때 평균한다.
with open("model/aux_features.json", "w") as f:
    json.dump(_aux_cols, f)
_oof, _, _auxms = _fit_aux(X_full[_aux_cols], _rev, None, "배포", _aux_params)
for _k, _m in enumerate(_auxms):
    _m.save_model("model/aux_rev_%d.cbm" % _k)

# 오프셋 측정용 별도 판: 검증 시즌(HOLDOUT_SEASON) 라벨을 **안 본** 보조 모델.
# 이걸 안 하면 오프셋 셀이 낙관적으로 측정돼 재중심화 상수가 어긋난다 (134점짜리다).
_hm = (df_processed["season"] <= HOLDOUT_SEASON - 1).to_numpy()
_vm = (df_processed["season"] == HOLDOUT_SEASON).to_numpy()
_o2, _v2, _ = _fit_aux(X_full.loc[_hm, _aux_cols].reset_index(drop=True), _rev[_hm],
                       X_full.loc[_vm, _aux_cols].reset_index(drop=True),
                       "오프셋용", _aux_params)
AUX_HO = np.full(len(X_full), np.nan)
AUX_HO[_hm] = _o2
AUX_HO[_vm] = _v2

X_full["aux_rev"] = _oof.astype(np.float32)
feature_cols = list(feature_cols) + ["aux_rev"]
print("  aux_rev 추가: OOF 평균 %.4f (실제 reverse 율 %.4f) | 피처 %d -> %d"
      % (np.nanmean(_oof), np.nanmean(_rev), len(feature_cols) - 1, len(feature_cols)))
assert not np.isnan(_oof).any(), "aux_rev 에 NaN 이 있다 -- 폴드가 전체를 안 덮었다"

# feature_cols 가 바뀌었으므로 selected_features.json 을 다시 쓴다.
# (원본 기록은 이 블록보다 앞에 있어서 aux_rev 가 빠져 있다.)
with open("model/selected_features.json", "w") as f:
    json.dump(list(feature_cols), f)
print("  selected_features.json 재기록 (%d개)" % len(feature_cols))
# =========================================================================



## [Cell 6b] 최종 학습 — fold 10 × seed 3 = 30개 모델

Optuna로 찾은 파라미터로 전체 데이터에 대해 학습한다. `StratifiedKFold`를 seed별로
독립적으로 3번 돌려서 (10-fold × 3-seed = 30개 모델), 최종 제출은 이 30개를 평균한다.
fold 수를 늘리고 seed를 다양화하는 건 각각 **분산을 줄이는 효과**라 안전하게 쌓인다.

⚠️ 모델 30개를 학습하므로 기존(5개)보다 시간이 훨씬 오래 걸린다 (GPU 기준 체감 3~6배).


In [ ]:
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import brier_score_loss
from sklearn.calibration import CalibratedClassifierCV
from catboost import CatBoostClassifier

X, y = X_full, y_full  # Cell 6a에서 만든 전체 데이터 재사용


def extract_isotonic(cv_obj):
    cc = cv_obj.calibrated_classifiers_[0]
    if hasattr(cc, 'calibrators'):
        return cc.calibrators[0]
    if hasattr(cc, 'calibrators_'):
        return cc.calibrators_[0]
    raise AttributeError("보정기를 찾을 수 없습니다.")


def brier_and_skill(y_t, p, tag=""):
    b = brier_score_loss(y_t, p)
    r = np.mean(y_t)
    naive = r * (1 - r)
    skill = 1 - b / naive
    print(f"  {tag:<20} Brier={b:.5f}  Skill={skill:+.3%}  (리더보드 환산 ≈ {skill*100000:,.0f})")
    return skill


seed_oof_raw = {s: np.zeros(len(X)) for s in SEEDS}
seed_oof_cal = {s: np.zeros(len(X)) for s in SEEDS}

print(f"\n=== 최종 학습: {N_SPLITS}-fold x {len(SEEDS)}-seed = {N_SPLITS * len(SEEDS)}개 모델 ===")
for seed in SEEDS:
    print(f"\n########## SEED {seed} ##########")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"[ seed {seed} / fold {fold+1}/{N_SPLITS} ]", end=" ")
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        params = dict(BEST_PARAMS)
        params["random_seed"] = seed
        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)
        raw = model.predict_proba(X_val)[:, 1]
        seed_oof_raw[seed][val_idx] = raw
        model.save_model(f"model/cb_fold_{seed}_{fold+1}.cbm")

        _c = CalibratedClassifierCV(model, method='isotonic', cv='prefit')
        _c.fit(X_val, y_val)
        iso = extract_isotonic(_c)
        cal = iso.predict(raw)
        seed_oof_cal[seed][val_idx] = cal
        joblib.dump(iso, f"model/isotonic_fold_{seed}_{fold+1}.pkl")

        print(f"Brier(cal)={brier_score_loss(y_val, cal):.5f}")

print(f"\n학습 및 저장 완료 ({N_SPLITS * len(SEEDS)}개 모델)")

print("\n=== seed별 전체 OOF ===")
for seed in SEEDS:
    brier_and_skill(y.to_numpy(), seed_oof_cal[seed], f"seed {seed}")

print("\n=== seed 평균 OOF (이게 최종 제출과 가장 비슷한 조합) ===")
avg_oof_cal = np.mean([seed_oof_cal[s] for s in SEEDS], axis=0)
brier_and_skill(y.to_numpy(), avg_oof_cal, "seed 평균")
print("\n주의: 이 OOF도 StratifiedKFold 기반이라 절대 성능 지표가 아니다.")
print("      933점(fold5/seed1 구성) 대비 개선 여부는 리더보드로만 판단할 것.")


## [Cell 6c] 재중심화 오프셋 산출 (test 미참조 고정 상수)

성공률이 매 시즌 단조 하락한다 (.5647 → .5327 → .5328 → .5289 → .5000 → .4861).
모델은 train 평균(≈.524) 쪽으로 캘리브레이션되므로 미래 시즌에서는 평균이 위로 뜬다.

**규정 주의**: 추론 시점에 "test 예측 평균이 목표값이 되도록" 오프셋을 푸는 방식은
평가 데이터 전체를 보고 만든 사후 보정값에 해당할 소지가 있다. 그래서 여기서는
**학습 시점에** 오프셋을 측정해 상수로 박아둔다 — 구조를 최종 모델과 동일하게 맞춘다:

| | 학습 | 예측 대상 |
|---|---|---|
| 오프셋 측정(여기) | ~2023 | 2024 (정답 알고 있음 → 정확한 시프트 계산 가능) |
| 실제 제출 | ~2024 | 2025 (같은 "Y까지 학습 → Y+1 예측" 구조) |

2024 홀드아웃 검증에서 11개 설정 전부 +11~22점(평균 +18)이었다.


In [ ]:
# 홀드아웃(=2023까지 학습 -> 2024 예측)으로 로짓 오프셋을 측정해 상수로 고정한다.
# X, y, cat_features, BEST_PARAMS, extract_isotonic 은 Cell 6a/6b 에서 정의됨.

def solve_logit_offset(p, target):
    """평균 예측이 target 이 되게 하는 로짓 공간 상수 시프트."""
    q = np.clip(p, 1e-6, 1 - 1e-6)
    lo = np.log(q / (1 - q))
    off = 0.0
    for _ in range(300):
        cur = 1.0 / (1.0 + np.exp(-(lo + off)))
        err = cur.mean() - target
        if abs(err) < 1e-9:
            break
        off -= err * 4.0
    return float(off)


RECENTER_OFFSET = 0.0
if RECENTER:
    _tr_m = (df_processed['season'] <= HOLDOUT_SEASON - 1).to_numpy()
    _va_m = (df_processed['season'] == HOLDOUT_SEASON).to_numpy()
    _Xh, _yh = X[_tr_m].copy(), y[_tr_m]
    _Xv, _yv = X[_va_m].copy(), y[_va_m]
    # 검증 시즌 라벨을 안 본 보조 예측으로 갈아끼운다. 안 그러면 오프셋이
    # 낙관적으로 측정돼 재중심화 상수가 어긋난다 (지금 재중심화는 +134 다).
    _Xh["aux_rev"] = AUX_HO[_tr_m].astype(np.float32)
    _Xv["aux_rev"] = AUX_HO[_va_m].astype(np.float32)
    print("  오프셋용 aux_rev 교체 완료 (검증 시즌 라벨 미접촉)")
    print(f"오프셋 측정: 학습 {len(_Xh):,}행(~{HOLDOUT_SEASON-1}) -> 검증 {len(_Xv):,}행({HOLDOUT_SEASON})")

    _skf = StratifiedKFold(n_splits=N_HOLDOUT_FOLDS, shuffle=True, random_state=SEEDS[0])
    _ps = []
    for _f, (_ti, _vi) in enumerate(_skf.split(_Xh, _yh)):
        _p = dict(BEST_PARAMS); _p["random_seed"] = SEEDS[0]
        _m = CatBoostClassifier(**_p)
        _m.fit(_Xh.iloc[_ti], _yh.iloc[_ti], eval_set=(_Xh.iloc[_vi], _yh.iloc[_vi]), verbose=0)
        _c = CalibratedClassifierCV(_m, method='isotonic', cv='prefit')
        _c.fit(_Xh.iloc[_vi], _yh.iloc[_vi])
        _ps.append(extract_isotonic(_c).predict(_m.predict_proba(_Xv)[:, 1]))
        print(f"  홀드아웃 fold {_f+1}/{N_HOLDOUT_FOLDS} 완료")

    _ph = np.mean(_ps, axis=0)
    _actual = float(_yv.mean())
    RECENTER_OFFSET = solve_logit_offset(_ph, _actual)

    _naive = _actual * (1 - _actual)
    _sk = lambda q: (1 - ((np.clip(q, 1e-6, 1-1e-6) - _yv.to_numpy()) ** 2).mean() / _naive) * 100000
    _qq = np.clip(_ph, 1e-6, 1-1e-6)
    _after = 1.0 / (1.0 + np.exp(-(np.log(_qq/(1-_qq)) + RECENTER_OFFSET)))
    print(f"\n  {HOLDOUT_SEASON} 실제평균={_actual:.4f} | 보정전 평균예측={_ph.mean():.4f}")
    print(f"  로짓 오프셋 = {RECENTER_OFFSET:+.4f}")
    print(f"  홀드아웃 환산점수: 보정전 {_sk(_ph):,.0f} -> 보정후 {_sk(_after):,.0f}  ({_sk(_after)-_sk(_ph):+,.0f})")

# train_constants.json 갱신 (script.py 가 이 상수를 그대로 더한다)
with open("model/train_constants.json", "w") as f:
    json.dump({"prior_mean": PRIOR_MEAN, "trackman_mode": TRACKMAN_MODE,
               "recenter_offset": RECENTER_OFFSET}, f)
# 오프셋 셀이 파일을 덮어쓰므로 ws_* 상수를 다시 넣는다
with open("model/train_constants.json", "r") as f:
    _tc = json.load(f)
_tc["ws_target_season"] = int(df_train["season"].max()) + 1
_tc["ws_league_mean"] = ws_next_season_mean(ws_season_means(df_train), _tc["ws_target_season"])
_tc["ws_rates"] = WS_RATES
_tc["ws_C"] = WS_C
_tc["wb_league_mean"] = ws_next_season_mean(wb_season_means(df_train), _tc["ws_target_season"])
_tc["wb_rates"] = WB_RATES
_tc["wb_C"] = WB_C
_pf_m = pf_season_means(df_train)
_tc["pf_league_cur"] = ws_next_season_mean(_pf_m, _tc["ws_target_season"])
_tc["pf_league_prev"] = {c: float(_pf_m[c][max(_pf_m[c])]) for c, _ in PF_SPEC}
_tc["pf_spec"] = [[c, w] for c, w in PF_SPEC]
with open("model/train_constants.json", "w") as f:
    json.dump(_tc, f)
assert "ws_league_mean" in json.load(open("model/train_constants.json")), "ws 상수 유실"
assert "wb_league_mean" in json.load(open("model/train_constants.json")), "wb 상수 유실"
assert "pf_league_cur" in json.load(open("model/train_constants.json")), "pf 상수 유실"
print(f"\ntrain_constants.json 저장: recenter_offset={RECENTER_OFFSET:+.4f}")


## [Cell 7] `script.py` 생성 및 zip 패키징

In [ ]:
SCRIPT_TEMPLATE = r"""import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import json
import traceback
import numpy as np
import pandas as pd
import joblib
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

# ================= 학습과 문자 단위로 동일한 전처리 =================
__STEPS__
# ====================================================================


def main():
    data_dir = None
    for path in ["data", "open", "./data", "./open", "open/data"]:
        if os.path.exists(os.path.join(path, "test.csv")):
            data_dir = path
            break
    if data_dir is None:
        raise FileNotFoundError("평가용 데이터를 찾을 수 없습니다.")

    df_test = pd.read_csv(os.path.join(data_dir, "test.csv"))
    row_ids = df_test['row_id'].copy() if 'row_id' in df_test.columns else df_test.index

    constants_path = os.path.join("model", "train_constants.json")
    if not os.path.exists(constants_path):
        raise RuntimeError("model/train_constants.json이 없습니다. prior_mean을 알 수 없어 중단합니다.")
    with open(constants_path, "r") as f:
        _tc = json.load(f)
    prior_mean = float(_tc["prior_mean"])

    df_proc = step1_basic_features(df_test)
    df_proc = step2_pitcher_role_features(df_proc)
    df_proc = step3_matchup_features(df_proc)
    df_proc = step4_refined_count_features(df_proc)
    df_proc = step5_pitches_per_inning(df_proc)
    df_proc = step6_combined_runner_features(df_proc)
    df_proc = step7_bayesian_smoothing(df_proc, prior_mean=prior_mean)
    df_proc = step8_batter_toughness_features(df_proc)
    df_proc = step9_garbage_time_features(df_proc)
    df_proc = step10_recent_form_momentum(df_proc)
    df_proc = step11_veteran_and_pressure_features(df_proc)
    df_proc = step12_first_pitch_tendency(df_proc)
    df_proc = step13_sac_fly_threat(df_proc)

    if 'count_advantage' not in df_proc.columns:
        b, s = df_proc['balls_before'], df_proc['strikes_before']
        p_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
        b_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
        neu = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
        df_proc['count_advantage'] = np.select([p_ahead, b_ahead, neu],
                                                ['Pitcher', 'Batter', 'Neutral'], default='None')

    # ---------- 트랙맨 병합 ----------
    # 학습 때 쓴 방식(train_constants.json의 trackman_mode)을 그대로 따라간다.
    #   asof  : merge_asof backward. 2025 test 행은 가장 최근(2024) 값을 받는다.
    #   exact : (season, month) 정확 일치 merge + fillna(0).
    #           트랙맨에 2025가 없으므로 test에서는 전부 0이 된다 (900점 버전의 동작).
    with open(constants_path, "r") as f:
        _const = json.load(f)
    trackman_mode = _const.get("trackman_mode", "asof")

    _NA = ['', 'NaN', 'nan', 'NULL', 'null', 'NA', 'N/A', 'n/a']
    fd_path = os.path.join("model", "feat_diff.csv")
    fs_path = os.path.join("model", "feat_speed.csv")
    fr_path = os.path.join("model", "feat_rp.csv")
    has_tm = all(os.path.exists(p) for p in [fd_path, fs_path, fr_path])

    if has_tm:
        # 'None'은 0-0/3-2 카운트를 뜻하는 실제 문자열인데 pandas 기본 설정은
        # 이를 NaN으로 읽어버린다. 그러면 by= 매칭이 전량 실패한다.
        feat_diff = pd.read_csv(fd_path, keep_default_na=False, na_values=_NA)
        feat_speed = pd.read_csv(fs_path, keep_default_na=False, na_values=_NA)
        feat_rp = pd.read_csv(fr_path, keep_default_na=False, na_values=_NA)
        feat_diff['count_advantage'] = feat_diff['count_advantage'].astype(str)
        rp_value_cols = [c for c in feat_rp.columns if c.startswith('past_')]

        if trackman_mode == "asof":
            df_proc['time_idx'] = df_proc['season'] * 100 + df_proc['game_month']
            df_proc['__orig'] = np.arange(len(df_proc))
            df_proc = df_proc.sort_values('time_idx')
            feat_diff = feat_diff.sort_values('time_idx')
            feat_speed = feat_speed.sort_values('time_idx')
            feat_rp = feat_rp.sort_values('time_idx')

            df_proc = pd.merge_asof(
                df_proc,
                feat_diff[['time_idx', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']],
                on='time_idx', by=['pitcher_id', 'count_advantage'], direction='backward')
            df_proc = pd.merge_asof(
                df_proc, feat_speed[['time_idx', 'pitcher_id', 'past_fb_speed_mean']],
                on='time_idx', by='pitcher_id', direction='backward')
            df_proc = pd.merge_asof(
                df_proc, feat_rp[['time_idx', 'pitcher_id'] + rp_value_cols],
                on='time_idx', by='pitcher_id', direction='backward')

            df_proc = df_proc.sort_values('__orig').drop(columns=['__orig', 'time_idx'])
        else:
            df_proc = pd.merge(
                df_proc,
                feat_diff[['season', 'game_month', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']],
                on=['season', 'game_month', 'pitcher_id', 'count_advantage'], how='left')
            df_proc = pd.merge(
                df_proc, feat_speed[['season', 'game_month', 'pitcher_id', 'past_fb_speed_mean']],
                on=['season', 'game_month', 'pitcher_id'], how='left')
            df_proc = pd.merge(
                df_proc, feat_rp[['season', 'game_month', 'pitcher_id'] + rp_value_cols],
                on=['season', 'game_month', 'pitcher_id'], how='left')
            for c in ['expected_control_difficulty', 'past_fb_speed_mean'] + rp_value_cols:
                if c in df_proc.columns:
                    df_proc[c] = df_proc[c].fillna(0)

    # ---------- 조건부 투수통계 병합 ----------
    # 학습 때 저장한 룩업 테이블을 그대로 붙인다. 'None'(0-0/3-2 카운트) 보존을 위해
    # na_values 를 반드시 명시해야 한다 (기본 read_csv 는 NaN 으로 읽어 매칭이 전량 실패).
    for _nm, _keys in [("cond_p",   ["pitcher_id"]),
                       ("cond_pc",  ["pitcher_id", "count_advantage"]),
                       ("cond_ph",  ["pitcher_id", "batter_hand"]),
                       ("cond_phc", ["pitcher_id", "batter_hand", "count_advantage"]),
                       ("cond_pb",  ["pitcher_id", "batter_id"])]:
        _cp = os.path.join("model", _nm + ".csv")
        if not os.path.exists(_cp):
            continue
        _ct = pd.read_csv(_cp, keep_default_na=False, na_values=_NA)
        for _k in _keys:
            if _ct[_k].dtype == object or df_proc[_k].dtype == object:
                _ct[_k] = _ct[_k].astype(str)
                df_proc[_k] = df_proc[_k].astype(str)
        _n_before = len(df_proc)
        df_proc = df_proc.merge(_ct, on=_keys, how="left")
        if len(df_proc) != _n_before:
            raise RuntimeError("%s 병합에서 행 수가 %d -> %d 로 변함 (테이블 키 중복)"
                               % (_nm, _n_before, len(df_proc)))


    # ---------- 당해 시즌 성적 복원 ----------
    # 학습과 같은 규칙: '대상 시즌이 시작될 때의 커리어 상태' 를 빼서 당해 시즌만 남긴다.
    # 학습은 (투수, 시즌) 첫 행에서, 추론은 train 마지막 시즌 마지막 행에서 그 값을 얻는다.
    _ws_rates = _tc.get("ws_rates")
    if _ws_rates:
        _wsC = float(_tc.get("ws_C", 100.0))
        _wslg = _tc["ws_league_mean"]
        _pp = pd.read_csv(os.path.join("model", "pitcher_prior.csv"),
                          keep_default_na=False, na_values=_NA)
        _pp["pitcher_id"] = _pp["pitcher_id"].astype(df_proc["pitcher_id"].dtype)
        _nb = len(df_proc)
        df_proc = df_proc.merge(_pp, on="pitcher_id", how="left")
        if len(df_proc) != _nb:
            raise RuntimeError("pitcher_prior 병합에서 행 수가 변함")
        _n = df_proc["asof_pitcher_n"].to_numpy(dtype="float64")
        _n0 = df_proc["w_n0"].fillna(0.0).to_numpy(dtype="float64")   # 신규 투수는 0
        _wn = np.maximum(_n - _n0, 0.0)
        df_proc["w_n"] = _wn
        df_proc["w_share"] = _wn / np.maximum(_n, 1.0)
        for _k in _ws_rates:
            _c = "asof_pitcher_%s_rate" % _k
            _x = (df_proc[_c].fillna(0).to_numpy(dtype="float64") * _n).round()
            _x0 = df_proc["w_x0__" + _c].fillna(0.0).to_numpy(dtype="float64")
            _wx = np.clip(_x - _x0, 0.0, _wn)
            _b = float(_wslg[_c])
            df_proc["w_" + _k] = (_wx + _b * _wsC) / (_wn + _wsC) - _b
        df_proc = df_proc.drop(columns=[c for c in df_proc.columns
                                        if c == "w_n0" or c.startswith("w_x0__")])
        print("당해시즌 복원 완료: 투구수 중앙값 %.0f / 신규투수 비율 %.1f%%"
              % (float(np.median(_wn)), 100.0 * float((_n0 == 0).mean())))


    # ---------- 타자측 당해 시즌 복원 ----------
    _wb_rates = _tc.get("wb_rates")
    if _wb_rates:
        _wbC = float(_tc.get("wb_C", 100.0))
        _wblg = _tc["wb_league_mean"]
        _bp = pd.read_csv(os.path.join("model", "batter_prior.csv"),
                          keep_default_na=False, na_values=_NA)
        _bp["batter_id"] = _bp["batter_id"].astype(df_proc["batter_id"].dtype)
        _nb2 = len(df_proc)
        df_proc = df_proc.merge(_bp, on="batter_id", how="left")
        if len(df_proc) != _nb2:
            raise RuntimeError("batter_prior 병합에서 행 수가 변함")
        _n = df_proc["asof_batter_n"].to_numpy(dtype="float64")
        _n0 = df_proc["wb_n0"].fillna(0.0).to_numpy(dtype="float64")
        _wn = np.maximum(_n - _n0, 0.0)
        df_proc["wb_n"] = _wn
        df_proc["wb_share"] = _wn / np.maximum(_n, 1.0)
        for _k in _wb_rates:
            _c = "asof_batter_%s_rate" % _k
            _x = (df_proc[_c].fillna(0).to_numpy(dtype="float64") * _n).round()
            _x0 = df_proc["wb_x0__" + _c].fillna(0.0).to_numpy(dtype="float64")
            _wx = np.clip(_x - _x0, 0.0, _wn)
            _b = float(_wblg[_c])
            df_proc["wb_" + _k] = (_wx + _b * _wbC) / (_wn + _wbC) - _b
        df_proc = df_proc.drop(columns=[c for c in df_proc.columns
                                        if c == "wb_n0" or c.startswith("wb_x0__")])
        print("타자 당해시즌 복원 완료: 타석수 중앙값 %.0f / 신규타자 비율 %.1f%%"
              % (float(np.median(_wn)), 100.0 * float((_n0 == 0).mean())))


    # ---------- prev-game 시즌 경계 보정 ----------
    # 반드시 당해 시즌 복원(w_n) 뒤에 와야 한다.
    _pf_spec = _tc.get("pf_spec")
    if _pf_spec:
        _pfc, _pfp = _tc["pf_league_cur"], _tc["pf_league_prev"]
        _pa = pd.read_csv(os.path.join("model", "pitcher_appearance.csv"),
                          keep_default_na=False, na_values=_NA)
        _pa["pitcher_id"] = _pa["pitcher_id"].astype(df_proc["pitcher_id"].dtype)
        _n3 = len(df_proc)
        df_proc = df_proc.merge(_pa, on="pitcher_id", how="left")
        if len(df_proc) != _n3:
            raise RuntimeError("pitcher_appearance 병합에서 행 수가 변함")
        _av = df_proc["pf_avg_pa"].to_numpy(dtype="float64")
        _med = float(np.nanmedian(_av))
        _av = np.where(np.isfinite(_av), _av, _med)          # 신규 투수는 중앙값
        _wn = df_proc["w_n"].to_numpy(dtype="float64")
        _gest = _wn / np.maximum(_av, 1.0)
        df_proc["pf_gest"] = _gest
        _seen = set()
        for _c, _w in _pf_spec:
            _cross = np.clip(_w - _gest, 0.0, _w) / _w
            if _w not in _seen:
                df_proc["pf_cross%d" % _w] = _cross
                _seen.add(_w)
            _a, _b = float(_pfc[_c]), float(_pfp[_c])
            _blend = _a * (1.0 - _cross) + _b * _cross
            df_proc["pf_" + _c.replace("asof_pitcher_", "")] = \
                df_proc[_c].to_numpy(dtype="float64") - _blend
        df_proc = df_proc.drop(columns=["pf_avg_pa"])
        print("prev-game 보정 완료: 작년분 섞인 행 "
              + " ".join("prev%d %.1f%%" % (_w, 100.0 * float(
                  (df_proc["pf_cross%d" % _w] > 0).mean())) for _w in (1, 3, 5)))

    df_proc = step14_convert_to_category(df_proc)

    # ---------- aux_rev: 보조 모델로 P(reverse|X) 예측 ----------
    # 학습 때 교차적합해 만든 피처다. 추론은 폴드 모델 3개의 평균을 쓴다.
    # 이 행의 입력만으로 계산되므로 "평가 데이터 각 행 독립" 규정을 만족한다.
    import glob as _glob
    _auxf = os.path.join("model", "aux_features.json")
    if os.path.exists(_auxf):
        with open(_auxf, "r") as f:
            _aux_cols = json.load(f)
        _apaths = sorted(_glob.glob(os.path.join("model", "aux_rev_*.cbm")))
        if not _apaths:
            raise RuntimeError("aux_features.json 은 있는데 aux_rev_*.cbm 이 없다")
        for _c in _aux_cols:
            if _c not in df_proc.columns:
                df_proc[_c] = np.nan
        _Xa = df_proc[_aux_cols].copy()
        for _c in _Xa.columns:
            if _Xa[_c].dtype.name in ["category", "object"]:
                _Xa[_c] = _Xa[_c].astype(str).astype("category")
        _acc = np.zeros(len(_Xa))
        for _p in _apaths:
            _am = CatBoostClassifier()
            _am.load_model(_p)
            _acc += _am.predict_proba(_Xa)[:, 1] / len(_apaths)
        df_proc["aux_rev"] = _acc.astype(np.float32)
        print("aux_rev 예측 완료: 모델 %d개 | 평균 %.4f" % (len(_apaths), _acc.mean()))

    with open("model/selected_features.json", "r") as f:
        selected_features = json.load(f)
    if "aux_rev" in selected_features and "aux_rev" not in df_proc.columns:
        raise RuntimeError("selected_features 에 aux_rev 가 있는데 "
                           "만들어지지 않았다 -- aux_features.json / "
                           "aux_rev_*.cbm 을 확인할 것")
    for col in selected_features:
        if col not in df_proc.columns:
            df_proc[col] = np.nan
    df_features = df_proc[selected_features].copy()

    # CatBoost는 cat_features에 실제 NaN을 허용하지 않는다 (학습과 동일 처리)
    for col in df_features.columns:
        if df_features[col].dtype.name in ['category', 'object']:
            df_features[col] = df_features[col].astype(str).astype('category')

    # ---------- 추론: 모델 전체 평균 ----------
    # StratifiedKFold(shuffle=True) x 여러 seed로 학습했으므로 모든 모델이
    # 대등한 실력을 가진다 -> 균등 평균이 순수한 분산 감소로 이어진다.
    # 파일명에서 seed/fold 조합을 실제로 스캔한다 (개수를 하드코딩하지 않음 ->
    # N_SPLITS/SEEDS를 나중에 바꿔도 script.py 수정이 필요 없다).
    import glob
    preds = []
    cb_paths = sorted(glob.glob(os.path.join("model", "cb_fold_*.cbm")))
    for cb_path in cb_paths:
        stem = os.path.splitext(os.path.basename(cb_path))[0]  # cb_fold_{seed}_{fold}
        suffix = stem[len("cb_fold_"):]  # {seed}_{fold}
        model = CatBoostClassifier()
        model.load_model(cb_path)
        names = list(model.feature_names_)
        df_in = df_features.copy()
        for col in names:
            if col not in df_in.columns:
                df_in[col] = np.nan
        raw = model.predict_proba(df_in[names])[:, 1]
        iso_path = os.path.join("model", "isotonic_fold_%s.pkl" % suffix)
        if os.path.exists(iso_path):
            raw = joblib.load(iso_path).predict(raw)
        preds.append(raw)

    if len(preds) == 0:
        raise RuntimeError(
            "모델을 하나도 로드하지 못했습니다. cwd=%s, model=%s"
            % (os.getcwd(), sorted(os.listdir('model')) if os.path.isdir('model') else '(없음)'))

    final_preds = np.mean(preds, axis=0)
    if np.isnan(final_preds).any():
        final_preds = np.nan_to_num(final_preds, nan=prior_mean)

    # ---------- 재중심화 ----------
    # 학습 시점에 홀드아웃(~Y-1 학습 -> Y 예측)으로 측정해 박아둔 고정 로짓 오프셋.
    # test 를 전혀 참조하지 않으므로 '평가 데이터 전체를 보고 만든 사후 보정값'이 아니다.
    _off = float(_const.get("recenter_offset", 0.0))
    if _off != 0.0:
        _q = np.clip(final_preds, 1e-6, 1 - 1e-6)
        final_preds = 1.0 / (1.0 + np.exp(-(np.log(_q / (1 - _q)) + _off)))

    final_preds = np.clip(final_preds, 0.01, 0.99)

    os.makedirs("output", exist_ok=True)
    submission = pd.DataFrame({"row_id": row_ids, "control_success": final_preds})

    sample_path = os.path.join(data_dir, "sample_submission.csv")
    if os.path.exists(sample_path):
        sample = pd.read_csv(sample_path)
        sample['row_id'] = sample['row_id'].astype(str)
        submission['row_id'] = submission['row_id'].astype(str)
        sample = sample.drop(columns=['control_success'], errors='ignore')
        sample = sample.merge(submission, on='row_id', how='left')
        sample['control_success'] = sample['control_success'].fillna(prior_mean)
        sample.to_csv("output/submission.csv", index=False)
    else:
        submission.to_csv("output/submission.csv", index=False)


if __name__ == "__main__":
    try:
        main()
    except Exception:
        os.makedirs("output", exist_ok=True)
        with open("output/error_log.txt", "w", encoding="utf-8") as f:
            f.write(traceback.format_exc())
        raise
"""

script_content = SCRIPT_TEMPLATE.replace("__STEPS__", STEPS_SRC)

with open("script.py", "w", encoding="utf-8") as f:
    f.write(script_content)
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("catboost\n")

import ast
ast.parse(script_content)
print(f"script.py 생성 완료 ({len(script_content):,}자, 문법 검사 통과)")


In [ ]:
import zipfile
import glob

cb_files = sorted(glob.glob("model/cb_fold_*.cbm"))
iso_files = sorted(glob.glob("model/isotonic_fold_*.pkl"))
expected_n = N_SPLITS * len(SEEDS)
print(f"모델 파일 {len(cb_files)}개 발견 (기대 {expected_n}개)")
if len(cb_files) != expected_n or len(iso_files) != expected_n:
    raise RuntimeError(
        f"모델 파일 개수가 예상과 다릅니다 (cb={len(cb_files)}, iso={len(iso_files)}, "
        f"기대={expected_n}). Cell 6b가 끝까지 정상 실행됐는지 확인하세요."
    )

REQUIRED = (
    ["script.py", "requirements.txt"]
    + [os.path.relpath(p) for p in cb_files]
    + [os.path.relpath(p) for p in iso_files]
    + ["model/selected_features.json", "model/train_constants.json", "model/best_params.json",
       "model/feat_diff.csv", "model/feat_speed.csv", "model/feat_rp.csv"]
    + [f"model/{n}.csv" for n in ["cond_p", "cond_pc", "cond_ph", "cond_phc", "cond_pb"]
       if os.path.exists(f"model/{n}.csv")]
    + (["model/pitcher_prior.csv"]
       if os.path.exists("model/pitcher_prior.csv") else [])
    + (["model/batter_prior.csv"]
       if os.path.exists("model/batter_prior.csv") else [])
    + (["model/pitcher_appearance.csv"]
       if os.path.exists("model/pitcher_appearance.csv") else [])
    + sorted(glob.glob("model/aux_rev_*.cbm"))
    + (["model/aux_features.json"]
       if os.path.exists("model/aux_features.json") else [])
)

missing = [p for p in REQUIRED if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"다음 파일이 없습니다: {missing}")

ZIP_PATH = "submit_v10wn.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in REQUIRED:
        zf.write(p, arcname=p)
print(f"{ZIP_PATH} 생성 완료 — {len(REQUIRED)}개 파일")


## [Cell 8] 파이프라인 버그 탐지 (성능 측정용 아님)

`script.py`를 실제로 실행해 `submission.csv`가 정상적으로 나오는지 확인한다.

**⚠️ 여기 나오는 Skill 수치는 성능 지표로 쓰면 안 된다.** `StratifiedKFold`로 학습해
모델이 이 홀드아웃 행들을 이미 학습에 사용했으므로, 점수가 크게 부풀려진다.

이 셀의 목적은 오직 다음 세 가지 확인이다.
1. 스크립트가 크래시 없이 끝까지 도는가
2. `submission.csv`의 행 수와 `row_id`가 맞는가
3. 예측 분포가 상식적인가 (`std`가 0에 가깝거나 전부 0.01이면 버그)


In [ ]:
import shutil, subprocess, sys

SANDBOX = "validation_sandbox"
_n = min(50000, len(df_train))
sample = df_train.sample(_n, random_state=1).reset_index(drop=True)

if os.path.exists(SANDBOX):
    shutil.rmtree(SANDBOX)
os.makedirs(f"{SANDBOX}/data", exist_ok=True)
sample.to_csv(f"{SANDBOX}/data/test.csv", index=False)
shutil.copy2("script.py", f"{SANDBOX}/script.py")
shutil.copytree("model", f"{SANDBOX}/model")

print(f"{_n:,}행으로 script.py 실행 중...")
res = subprocess.run([sys.executable, "script.py"], cwd=SANDBOX, capture_output=True, text=True)
print("종료 코드:", res.returncode)
if res.stderr.strip():
    print("--- stderr ---")
    print(res.stderr[-3000:])

sub_path = f"{SANDBOX}/output/submission.csv"
if not os.path.exists(sub_path):
    err = f"{SANDBOX}/output/error_log.txt"
    if os.path.exists(err):
        print(open(err, encoding="utf-8").read())
    raise RuntimeError("submission.csv가 생성되지 않았습니다.")

sub = pd.read_csv(sub_path)
p = sub["control_success"].to_numpy()
print(f"\n행 수: {len(sub):,} (기대 {_n:,})  결측: {np.isnan(p).sum()}")
print(f"예측 분포: min={p.min():.4f} max={p.max():.4f} mean={p.mean():.4f} std={p.std():.4f}")

ok = (len(sub) == _n) and (np.isnan(p).sum() == 0) and (p.std() > 0.005)
print("\n[통과] 파이프라인 정상." if ok else "\n[실패] 위 수치를 확인하세요.")
print("(반복: 이 셀은 버그 탐지용이며 성능 판단용이 아닙니다.)")


---

## 실행 및 제출 전략

1. Cell 0~7 순서대로 실행 (Cell 6a: 튜닝, Cell 6b: 최종 30개 모델 학습 — 시간이 꽤 걸림)
2. `submit_tuned.zip` 제출해서 933점 대비 개선 여부 확인
3. 개선됐으면 이 구성을 새 기준선으로 삼고, 다음 단계(`asof_*` 피처 재설계)로 진행

## 그 다음 개선 후보

| 순위 | 항목 | 이유 |
|---|---|---|
| 1 | **`asof_*` 피처 직접 재설계** | 카운트/구종/좌우별 세분화, 경기 단위가 아닌 최근 N구 이동평균. 작업량은 크지만 로드맵상 핵심 승부처 |
| 2 | **LightGBM 재투입** | 이번엔 CatBoost와 대등한 조건(StratifiedKFold)에서 블렌드 비율을 OOF로 탐색 |
| 3 | **`middle_rate`/`reverse_rate` 분해 기반 보조모델** | 제구 실패의 3가지 유형(가운데/크게벗어남/반대방향)을 따로 예측해서 스태킹 |

마감(9/2)까지 최소 이틀은 신규 시도 없이 버퍼로 남겨둘 것.
